# Dataset Preparation

## Utils

In [6]:

def add_length(example):
    text = str(example.get("text") or "")

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    return {
        "raw_token_length": len(token_ids),
    }

def find_column(dataset, candidates):
    for column in candidates:
        if column in dataset.column_names:
            return column

    raise KeyError(
        f"None of {candidates} found. "
        f"Available columns: {dataset.column_names}"
    )

def inspect_pairs_by_document_length(
    min_tokens: int,
    max_tokens: int,
    n: int = 20,
    seed: int = 42,
) -> None:
    matching_qrels = []

    for qrel in qrels:
        query_id = str(qrel[qrel_query_column])
        document_id = str(qrel[qrel_corpus_column])

        document = corpus_by_id.get(document_id)
        query = queries_by_id.get(query_id)

        if document is None or query is None:
            continue

        length = document["raw_token_length"]

        if min_tokens <= length <= max_tokens:
            matching_qrels.append(
                {
                    "query_id": query_id,
                    "document_id": document_id,
                    "query": str(query["text"]),
                    "document": str(document["text"]),
                    "title": str(document.get("title") or ""),
                    "raw_token_length": length,
                    "score": qrel.get("score", 1),
                }
            )

    print(
        f"{len(matching_qrels)} relevant query-document pairs with "
        f"documents between {min_tokens} and {max_tokens} tokens"
    )

    if not matching_qrels:
        return

    sample = (
        Dataset.from_list(matching_qrels)
        .shuffle(seed=seed)
        .select(range(min(n, len(matching_qrels))))
    )

    for row in sample:
        print("=" * 100)
        print("Query ID:", row["query_id"])
        print("Document ID:", row["document_id"])
        print("Document length:", row["raw_token_length"])
        print("Relevance score:", row["score"])
        print("Query:", row["query"])
        print("Title:", row["title"])
        print("Document:", repr(row["document"]))

## CulturaViva

In [16]:
from datasets import load_dataset, Dataset, DatasetDict

raw = load_dataset("DeepMount00/CulturaViva-ITA", split="train")
split_ds = raw.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"]
test_ds = split_ds["test"]


def build_ir_split(ds, split_name):
    queries = []
    corpus = []
    qrels = []

    for i, row in enumerate(ds):
        qid = f"q_{i}"
        did = f"d_{i}"

        queries.append(
            {
                "_id": qid,
                "text": row["prompt"],
            }
        )

        corpus.append(
            {
                "_id": did,
                "title": "",
                "text": row["answer"],
            }
        )

        qrels.append(
            {
                "query-id": qid,
                "corpus-id": did,
                "score": 1,
            }
        )

    return (
        Dataset.from_list(queries),
        Dataset.from_list(corpus),
        Dataset.from_list(qrels),
    )


queries_train, corpus_train, qrels_train = build_ir_split(train_ds, "train")
queries_test, corpus_test, qrels_test = build_ir_split(test_ds, "test")

repo_id = "lopozz/CulturaViva-Retrieval"

# push to repo
DatasetDict({
    "train": queries_train,
    "test": queries_test,
}).push_to_hub(repo_id, config_name="queries")

# push corpus as one config
DatasetDict({
    "train": corpus_train,
    "test": corpus_test,
}).push_to_hub(repo_id, config_name="corpus")

# push qrels as one config
DatasetDict({
    "train": qrels_train,
    "test": qrels_test,
}).push_to_hub(repo_id, config_name="qrels")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


... (22 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/lopozz/CulturaViva-Retrieval/commit/0458679152af17a5c8f4f811dccf9928ed0c993c', commit_message='Upload dataset', commit_description='', oid='0458679152af17a5c8f4f811dccf9928ed0c993c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/lopozz/CulturaViva-Retrieval', endpoint='https://huggingface.co', repo_type='dataset', repo_id='lopozz/CulturaViva-Retrieval'), pr_revision=None, pr_num=None)

## deepset/germandpr

In [15]:
from pathlib import Path

import httpx
from tqdm.auto import tqdm
from transformers import AutoTokenizer

VLLM_MODEL = "google/translategemma-12b-it"
VLLM_API_KEY="0cb0d80524dd713485412a923c51798295b01eca66ce8d0a3069377fbab10d1a"

tokenizer = AutoTokenizer.from_pretrained(VLLM_MODEL)

def build_translation_prompt(text: str, source_lang_code: str, target_lang_code: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "source_lang_code": source_lang_code,
                    "target_lang_code": target_lang_code,
                    "text": str(text or "").strip(),
                    "image": None,
                }
            ],
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def translate_texts(
    texts: list[str],
    source_lang_code: str,
    target_lang_code: str,
    max_new_tokens: int,
    client: httpx.Client,
) -> list[str]:
    if not texts:
        return []

    prompts = [
        build_translation_prompt(text, source_lang_code, target_lang_code)
        for text in texts
    ]

    response = client.post(
        "http://localhost:8000/v1/completions",
        json={
            "model": VLLM_MODEL,
            "prompt": prompts,
            "temperature": 0,
            "max_tokens": max_new_tokens,
        },
    )
    response.raise_for_status()

    data = response.json()

    choices = sorted(
        data["choices"],
        key=lambda choice: choice["index"],
    )

    translations = [
        str(choice["text"]).strip()
        for choice in choices
    ]

    if len(translations) != len(texts):
        raise RuntimeError(
            "Translation count mismatch: "
            f"sent {len(texts)}, received {len(translations)}"
        )

    return translations


def translate_dataset_texts(
    texts: list[str],
    source_lang_code: str,
    target_lang_code: str,
    request_batch_size: int,
    max_new_tokens: int,
) -> list[str]:
    translations: list[str] = []

    headers = {
        "Authorization": f"Bearer {VLLM_API_KEY}",
    }

    with httpx.Client(
        headers=headers,
        timeout=600,
    ) as client:
        for start in tqdm(
            range(0, len(texts), request_batch_size),
            desc="Translating",
        ):
            batch = texts[start : start + request_batch_size]

            batch_translations = translate_texts(
                texts=batch,
                source_lang_code=source_lang_code,
                target_lang_code=target_lang_code,
                max_new_tokens=max_new_tokens,
                client=client,
            )

            translations.extend(batch_translations)

    return translations

In [16]:
from pathlib import Path

from datasets import Dataset, DatasetDict

data_dir = Path("../data/germandpr_mteb").resolve()

expected_files = [
    "train_queries.parquet",
    "test_queries.parquet",
    "train_corpus.parquet",
    "test_corpus.parquet",
    "train_qrels.parquet",
    "test_qrels.parquet",
]

missing = [
    filename
    for filename in expected_files
    if not (data_dir / filename).is_file()
]

if missing:
    available = sorted(
        str(path.relative_to(data_dir))
        for path in data_dir.rglob("*")
        if path.is_file()
    )

    raise FileNotFoundError(
        f"Missing files: {missing}\n"
        f"Data directory: {data_dir}\n"
        f"Available files:\n" + "\n".join(available)
    )


def load_parquet(filename: str) -> Dataset:
    return Dataset.from_parquet(str(data_dir / filename))


queries = DatasetDict(
    {
        "train": load_parquet("train_queries.parquet"),
        "test": load_parquet("test_queries.parquet"),
    }
)

corpus = DatasetDict(
    {
        "train": load_parquet("train_corpus.parquet"),
        "test": load_parquet("test_corpus.parquet"),
    }
)

qrels = DatasetDict(
    {
        "train": load_parquet("train_qrels.parquet"),
        "test": load_parquet("test_qrels.parquet"),
    }
)

In [17]:
from pathlib import Path

from datasets import DatasetDict

SOURCE_LANGUAGE = "de"
TARGET_LANGUAGE = "it"
OUTPUT_DIR = Path("../data/germandpr_itquad").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def replace_text_column(dataset, translations):
    if len(dataset) != len(translations):
        raise ValueError(
            f"Length mismatch: dataset has {len(dataset)} rows, "
            f"but received {len(translations)} translations."
        )

    return dataset.remove_columns("text").add_column(
        "text",
        translations,
    )


translated_corpus = DatasetDict()
translated_queries = DatasetDict()

for split in ("train", "test"):
    print(f"\nTranslating corpus split: {split}")

    corpus_texts = translate_dataset_texts(
        corpus[split]["text"],
        source_lang_code=SOURCE_LANGUAGE,
        target_lang_code=TARGET_LANGUAGE,
        request_batch_size=32,
        max_new_tokens=2000,
    )

    translated_corpus[split] = replace_text_column(
        corpus[split],
        corpus_texts,
    )

    print(f"\nTranslating query split: {split}")

    query_texts = translate_dataset_texts(
        queries[split]["text"],
        source_lang_code=SOURCE_LANGUAGE,
        target_lang_code=TARGET_LANGUAGE,
        request_batch_size=32,
        max_new_tokens=128,
    )

    translated_queries[split] = replace_text_column(
        queries[split],
        query_texts,
    )

    translated_corpus[split].to_parquet(
        str(OUTPUT_DIR / f"{split}_corpus.parquet")
    )

    translated_queries[split].to_parquet(
        str(OUTPUT_DIR / f"{split}_queries.parquet")
    )

    # IDs remain unchanged, so qrels do not need translation.
    qrels[split].to_parquet(
        str(OUTPUT_DIR / f"{split}_qrels.parquet")
    )

print(f"Saved translated dataset to: {OUTPUT_DIR}")


Translating corpus split: train


Translating:   9%|▊         | 64/751 [42:28<7:35:58, 39.82s/it]


KeyboardInterrupt: 

In [18]:
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer


DATA_DIR = Path("../data/germandpr_itquad_dryrun").resolve()
SPLIT = "train"

corpus = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_corpus.parquet")
)

queries = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_queries.parquet")
)

qrels = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_qrels.parquet")
)

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)


corpus_with_lengths = corpus.map(
    add_length,
    desc="Computing document lengths",
)

corpus_id_column = find_column(
    corpus_with_lengths,
    ("_id", "id", "corpus-id", "corpus_id"),
)

query_id_column = find_column(
    queries,
    ("_id", "id", "query-id", "query_id"),
)

qrel_query_column = find_column(
    qrels,
    ("query-id", "query_id", "query-id"),
)

qrel_corpus_column = find_column(
    qrels,
    ("corpus-id", "corpus_id", "corpus-id"),
)


corpus_by_id = {
    str(row[corpus_id_column]): row
    for row in corpus_with_lengths
}

queries_by_id = {
    str(row[query_id_column]): row
    for row in queries
}

Generating train split: 10 examples [00:00, 2677.84 examples/s]
Generating train split: 10 examples [00:00, 3401.98 examples/s]
Generating train split: 9275 examples [00:00, 1872186.80 examples/s]
Computing document lengths: 100%|██████████| 10/10 [00:00<00:00, 389.15 examples/s]


In [ ]:
# inspect_pairs_by_document_length(1, 198) # no

# inspect_pairs_by_document_length(199, 400) # buco
# inspect_pairs_by_document_length(339, 450) # 110
# inspect_pairs_by_document_length(450, 500) # 315
# inspect_pairs_by_document_length(500, 1000) # 616

3 relevant query-document pairs with documents between 199 and 400 tokens
Query ID: q2
Document ID: c2
Document length: 300
Relevance score: 1
Query: Quanto spazio di archiviazione ha l'iPod di seconda generazione?
Title: IPod
Document: 'iPod\n\n==== Seconda generazione ====\nIl 12 settembre 2006, Steve Jobs ha presentato l\'iPod shuffle di seconda generazione. Dal 5 settembre 2007, l\'iPod shuffle è disponibile in nuove colorazioni, che corrispondono a quelle dell\'iPod nano di terza generazione. Per la prima volta, anche il modello più piccolo della gamma iPod è disponibile nella "Product Red Series". Dal 19 febbraio 2008, oltre al modello da 1 GB, l\'iPod shuffle è stato disponibile anche in una versione da 2 GB. Il 9 settembre 2008, i colori disponibili per lo Shuffle sono stati modificati. Insieme al dispositivo, è inclusa una piccola base di ricarica a batteria che si collega tramite la presa per le cuffie dell\'iPod shuffle. Sul retro, sono presenti due piccoli interruttori util

### Push to Hub

In [13]:
from datasets import Dataset

TARGET_REPO_ID = "lopozz/GermanDPRRetrieval"
DE_DIR = Path("../data/germandpr_mteb").resolve()
IT_DIR = Path("../data/germandpr_itquad").resolve()
SPLITS = ("train", "test")

for split in SPLITS:
    for lang, data_dir in (("de", DE_DIR), ("it", IT_DIR)):
        for subset in ("corpus", "queries", "qrels"):
            ds = Dataset.from_parquet(str(data_dir / f"{split}_{subset}.parquet"))
            ds.push_to_hub(TARGET_REPO_ID, config_name=f"{lang}-{subset}", split=split)

print(f"Pushed to: https://huggingface.co/datasets/{TARGET_REPO_ID}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


... (52 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to: https://huggingface.co/datasets/lopozz/GermanDPRRetrieval


## NorQuAD

In [8]:
from transformers import AutoTokenizer
import requests
model_name = "google/translategemma-12b-it"
tok = AutoTokenizer.from_pretrained(model_name)
VLLM_API_KEY="0cb0d80524dd713485412a923c51798295b01eca66ce8d0a3069377fbab10d1a"

messages = [{
    "role": "user",
    "content": [{
        "type": "text",
        "source_lang_code": "no",
        "target_lang_code": "it",
        "text": "Hva er hovedstaden i Norge?",
        "image": None,
    }],
}]

prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

response = requests.post(
    "http://localhost:8000/v1/completions",
    headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
    json={"model": model_name, "prompt": prompt, "temperature": 0, "max_tokens": 64},
)

response.raise_for_status()

translation = response.json()["choices"][0]['text']
print(translation)

Qual è la capitale della Norvegia?


In [3]:
from __future__ import annotations
from datasets import load_dataset

REPO_ID = "mteb/NorQuadRetrieval"
SOURCE_LANGUAGE = "no"
TARGET_LANGUAGE = "it"
OUTPUT_DIR = Path("../data/norquad_itquad")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ("train", "test", "val"):

    corpus = load_dataset(
        REPO_ID,
        "corpus",
        split=split,
    )

    queries = load_dataset(
        REPO_ID,
        "queries",
        split=split,
    )

    qrels = load_dataset(
        REPO_ID,
        "qrels",
        split=split,
    )

    print(f"\nTranslating corpus split: {split}")
    translated_corpus_texts = translate_dataset_texts(
        corpus["text"],
        source_lang_code=SOURCE_LANGUAGE,
        target_lang_code=TARGET_LANGUAGE,
        request_batch_size=32,
        max_new_tokens=1200,
    )

    translated_corpus = corpus.remove_columns("text").add_column(
        "text",
        translated_corpus_texts,
    )

    print(f"\nTranslating query split: {split}")
    translated_query_texts = translate_dataset_texts(
        queries["text"],
        source_lang_code=SOURCE_LANGUAGE,
        target_lang_code=TARGET_LANGUAGE,
        request_batch_size=32,
        max_new_tokens=500,
    )

    translated_queries = queries.remove_columns("text").add_column(
        "text",
        translated_query_texts,
    )


    translated_corpus.to_parquet(
        str(OUTPUT_DIR / f"{split}_corpus.parquet")
    )

    translated_queries.to_parquet(
        str(OUTPUT_DIR / f"{split}_queries.parquet")
    )

    qrels.to_parquet(
        str(OUTPUT_DIR / f"{split}_qrels.parquet")
    )

print(f"Saved translated dataset to: {OUTPUT_DIR}")


Translating corpus split: train


Translating: 100%|██████████| 50/50 [10:50<00:00, 13.00s/it]



Translating query split: train


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1721.09ba/s]



Translating corpus split: val


Translating: 100%|██████████| 10/10 [01:55<00:00, 11.51s/it]



Translating query split: val


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1269.46ba/s]

Saved translated dataset to: ../data/norquad_itquad


In [ ]:
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer


DATA_DIR = Path("../data/norquad_itquad").resolve()
SPLIT = "train"

corpus = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_corpus.parquet")
)

queries = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_queries.parquet")
)

qrels = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_qrels.parquet")
)

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)


corpus_with_lengths = corpus.map(
    add_length,
    desc="Computing document lengths",
)

corpus_id_column = find_column(
    corpus_with_lengths,
    ("_id", "id", "corpus-id", "corpus_id"),
)

query_id_column = find_column(
    queries,
    ("_id", "id", "query-id", "query_id"),
)

qrel_query_column = find_column(
    qrels,
    ("query-id", "query_id", "query-id"),
)

qrel_corpus_column = find_column(
    qrels,
    ("corpus-id", "corpus_id", "corpus-id"),
)


corpus_by_id = {
    str(row[corpus_id_column]): row
    for row in corpus_with_lengths
}

queries_by_id = {
    str(row[query_id_column]): row
    for row in queries
}

Generating train split: 1586 examples [00:00, 213545.83 examples/s]
Generating train split: 1024 examples [00:00, 539704.36 examples/s]
Generating train split: 2048 examples [00:00, 969956.48 examples/s]
Computing document lengths: 100%|██████████| 1586/1586 [00:00<00:00, 2712.13 examples/s]


In [44]:
# inspect_pairs_by_document_length(1, 3) # no
# inspect_pairs_by_document_length(4, 10) # no
# inspect_pairs_by_document_length(11, 20) # no
# inspect_pairs_by_document_length(20, 66) # 58
# inspect_pairs_by_document_length(67, 338) # buco
# inspect_pairs_by_document_length(339, 450) # 110
# inspect_pairs_by_document_length(450, 500) # 315
# inspect_pairs_by_document_length(500, 1000) # 616

#### Push to Hub

In [10]:
from datasets import Dataset, load_dataset

REPO_ID = "mteb/NorQuadRetrieval"
TARGET_REPO_ID = "lopozz/NorQuadRetrieval"
OUTPUT_DIR = Path("../data/norquad_itquad")
SPLITS = ("train", "test", "val")


def with_underscore_id(ds: Dataset) -> Dataset:
    if "id" in ds.column_names and "_id" not in ds.column_names:
        ds = ds.rename_column("id", "_id")
    return ds


for split in SPLITS:
    # Original Norwegian
    for subset in ("corpus", "queries"):
        ds = with_underscore_id(load_dataset(REPO_ID, subset, split=split))
        ds.push_to_hub(TARGET_REPO_ID, config_name=f"no-{subset}", split=split)

    no_qrels = load_dataset(REPO_ID, "qrels", split=split)
    no_qrels.push_to_hub(TARGET_REPO_ID, config_name="no-qrels", split=split)

    # Italian translation
    translated_corpus = with_underscore_id(Dataset.from_parquet(str(OUTPUT_DIR / f"{split}_corpus.parquet")))
    translated_corpus.push_to_hub(TARGET_REPO_ID, config_name="it-corpus", split=split)

    translated_queries = with_underscore_id(Dataset.from_parquet(str(OUTPUT_DIR / f"{split}_queries.parquet")))
    translated_queries.push_to_hub(TARGET_REPO_ID, config_name="it-queries", split=split)

    # qrels are untranslated (ids only), so it-qrels == no-qrels; reuse the parquet you already saved
    qrels = Dataset.from_parquet(str(OUTPUT_DIR / f"{split}_qrels.parquet"))
    qrels.push_to_hub(TARGET_REPO_ID, config_name="it-qrels", split=split)

print(f"Pushed to: https://huggingface.co/datasets/{TARGET_REPO_ID}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


... (89 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to: https://huggingface.co/datasets/lopozz/NorQuadRetrieval


## mMARCO

In [1]:
import ir_datasets
from datasets import Dataset
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Push to hub

In [1]:
import ir_datasets
from datasets import Dataset, DatasetDict, Features, Value, load_dataset
from huggingface_hub import login
from tqdm.auto import tqdm


REPO_ID = "lopozz/MMarcoRetrieval"
REFERENCE_REPO_ID = "mteb/MMarcoRetrieval"
PRIVATE = False

SOURCE_LANGUAGES = {
    "ar": "ar",
    "de": "de",
    "nl": "dt",
    "es": "es",
    "fr": "fr",
    "hi": "hi",
    "id": "id",
    "it": "it",
    "ja": "ja",
    "pt": "pt",
    "ru": "ru",
    "vi": "vi",
    "zh": "zh",
}

LANGUAGES = {
    language: {
        "corpus": f"mmarco/v2/{source_language}",
        "dev": f"mmarco/v2/{source_language}/dev/small",
    }
    for language, source_language in SOURCE_LANGUAGES.items()
}


login()


# ---------------------------------------------------------
# Load the canonical C-MTEB sampled corpus IDs
# ---------------------------------------------------------

reference_corpus = load_dataset(
    REFERENCE_REPO_ID,
    "corpus",
    split="dev",
)

print(reference_corpus)
print("Reference columns:", reference_corpus.column_names)

sampled_document_ids = [
    str(document_id)
    for document_id in reference_corpus["_id"]
]

assert len(sampled_document_ids) == 106_813
assert len(sampled_document_ids) == len(set(sampled_document_ids))

sampled_document_id_set = set(sampled_document_ids)

print("Canonical sampled documents:", len(sampled_document_ids))


# ---------------------------------------------------------
# Hugging Face schemas
# ---------------------------------------------------------

corpus_features = Features(
    {
        "id": Value("string"),
        "title": Value("string"),
        "text": Value("string"),
    }
)

query_features = Features(
    {
        "id": Value("string"),
        "text": Value("string"),
    }
)

qrels_features = Features(
    {
        "query-id": Value("string"),
        "corpus-id": Value("string"),
        "score": Value("int64"),
    }
)


# ---------------------------------------------------------
# Build and upload each aligned language subset
# ---------------------------------------------------------

for language, source_ids in LANGUAGES.items():
    print(f"\n{'=' * 60}")
    print(f"Processing: {language}")
    print(f"{'=' * 60}")

    corpus_source = ir_datasets.load(source_ids["corpus"])
    dev_source = ir_datasets.load(source_ids["dev"])

    docs_store = corpus_source.docs_store()

    # Preserve the same order as the existing C-MTEB corpus.
    def generate_sampled_corpus():

        for document_id in tqdm(
            sampled_document_ids,
            desc=f"Loading {language} corpus",
        ):

            document = docs_store.get(document_id)
            yield {
                "id": str(document.doc_id),
                "title": "",
                "text": document.text,
            }

    def generate_dev_queries():
        for query in dev_source.queries_iter():
            yield {
                "id": str(query.query_id),
                "text": query.text,
            }

    def generate_dev_qrels():
        for qrel in dev_source.qrels_iter():
            yield {
                "query-id": str(qrel.query_id),
                "corpus-id": str(qrel.doc_id),
                "score": int(qrel.relevance),
            }

    corpus = Dataset.from_generator(
        generate_sampled_corpus,
        features=corpus_features,
    )

    dev_queries = Dataset.from_generator(
        generate_dev_queries,
        features=query_features,
    )


    dev_qrels = Dataset.from_generator(
        generate_dev_qrels,
        features=qrels_features,
    )

    # -----------------------------------------------------
    # Validation
    # -----------------------------------------------------

    corpus_ids = set(corpus["id"])
    dev_query_ids = set(dev_queries["id"])

    dev_qrel_query_ids = set(dev_qrels["query-id"])
    dev_qrel_document_ids = set(dev_qrels["corpus-id"])

    assert len(corpus) == 106_813
    assert corpus_ids == sampled_document_id_set

    assert dev_qrel_query_ids <= dev_query_ids

    missing_relevant_documents = dev_qrel_document_ids - corpus_ids

    assert not missing_relevant_documents, (
        f"{language}: {len(missing_relevant_documents)} relevant "
        f"documents are absent from the sampled corpus. Examples: "
        f"{list(missing_relevant_documents)[:10]}"
    )

    print("Corpus:", len(corpus))
    print("Dev queries:", len(dev_queries))
    print("Dev qrels:", len(dev_qrels))
    print("Relevant dev documents missing:", 0)

    # -----------------------------------------------------
    # Hub configurations
    # -----------------------------------------------------

    corpus_dataset = DatasetDict(
        {
            "dev": corpus,
        }
    )

    queries_dataset = DatasetDict(
        {
            "dev": dev_queries,
        }
    )

    qrels_dataset = DatasetDict(
        {
            "dev": dev_qrels,
        }
    )

    corpus_dataset.push_to_hub(
        REPO_ID,
        config_name=f"{language}-corpus",
        private=PRIVATE,
        commit_message=(
            f"Align {language} corpus with C-MTEB sampled document set"
        ),
    )

    queries_dataset.push_to_hub(
        REPO_ID,
        config_name=f"{language}-queries",
        private=PRIVATE,
        commit_message=f"Upload {language} mMARCO v2 queries",
    )

    qrels_dataset.push_to_hub(
        REPO_ID,
        config_name=f"{language}-qrels",
        private=PRIVATE,
        commit_message=f"Upload {language} mMARCO v2 qrels",
    )

    print(f"Finished: {language}")


print(f"\nDataset updated: https://huggingface.co/datasets/{REPO_ID}")

Dataset({
    features: ['_id', 'text', 'title'],
    num_rows: 106813
})
Reference columns: ['_id', 'text', 'title']
Canonical sampled documents: 106813

Processing: ar


Generating train split: 0 examples [00:00, ? examples/s]

Loading ar corpus:   0%|          | 0/106813 [00:00<?, ?it/s]

[INFO] [starting] building docstore



                                                                          [INFO] If you have a local copy of https://huggingface.co/datasets/unicamp-dl/mmarco/resolve/main/data/google/collections/arabic_collection.tsv, you can symlink it here to avoid downloading it again: /home/lpozzi/.ir_datasets/downloads/b73406ce3a3d31edea240603d031be7a



                                                                          [INFO] [starting] https://huggingface.co/datasets/unicamp-dl/mmarco/resolve/main/data/google/collections/arabic_collection.tsv








































































































































































































































































































                                                                                                                      

KeyboardInterrupt: 

In [4]:
from datasets import load_dataset


REPO_ID = "lopozz/MMarcoRetrieval"
REFERENCE_REPO_ID = "mteb/MMarcoRetrieval"

LANGUAGES = ["it", "es", "zh"]
SPLIT = "dev"

# Use token=True if your repository is still private.
TOKEN = None  # Change to True for a private repository.


def load_component(repo_id, config, split):
    return load_dataset(
        repo_id,
        config,
        split=split,
        token=TOKEN,
    )


def get_id_column(dataset):
    if "_id" in dataset.column_names:
        return "_id"
    if "id" in dataset.column_names:
        return "id"

    raise ValueError(
        f"Could not find an ID column in {dataset.column_names}"
    )


def show_difference(name, left, right, left_label, right_label):
    only_left = left - right
    only_right = right - left

    print(f"\n{name}")
    print(f"  {left_label}: {len(left):,}")
    print(f"  {right_label}: {len(right):,}")
    print(f"  Only in {left_label}: {len(only_left):,}")
    print(f"  Only in {right_label}: {len(only_right):,}")

    if only_left:
        print(f"  Examples only in {left_label}: {list(only_left)[:10]}")

    if only_right:
        print(f"  Examples only in {right_label}: {list(only_right)[:10]}")

    return not only_left and not only_right


# ---------------------------------------------------------
# Load your multilingual dataset
# ---------------------------------------------------------

data = {}

for language in LANGUAGES:
    print(f"Loading {language}...")

    corpus = load_component(
        REPO_ID,
        f"{language}-corpus",
        SPLIT,
    )
    queries = load_component(
        REPO_ID,
        f"{language}-queries",
        SPLIT,
    )
    qrels = load_component(
        REPO_ID,
        f"{language}-qrels",
        SPLIT,
    )

    corpus_id_column = get_id_column(corpus)
    query_id_column = get_id_column(queries)

    corpus_ids_list = [str(value) for value in corpus[corpus_id_column]]
    query_ids_list = [str(value) for value in queries[query_id_column]]

    corpus_ids = set(corpus_ids_list)
    query_ids = set(query_ids_list)

    qrel_rows = {
        (
            str(row["query-id"]),
            str(row["corpus-id"]),
            int(row["score"]),
        )
        for row in qrels
    }

    qrel_query_ids = {
        str(value)
        for value in qrels["query-id"]
    }

    qrel_corpus_ids = {
        str(value)
        for value in qrels["corpus-id"]
    }

    # No duplicate query or document IDs.
    assert len(corpus_ids_list) == len(corpus_ids), (
        f"{language}: duplicate corpus IDs detected"
    )
    assert len(query_ids_list) == len(query_ids), (
        f"{language}: duplicate query IDs detected"
    )

    # Every dev qrel must reference an available query and document.
    missing_queries = qrel_query_ids - query_ids
    missing_documents = qrel_corpus_ids - corpus_ids

    assert not missing_queries, (
        f"{language}: qrels reference {len(missing_queries)} "
        f"missing queries. Examples: {list(missing_queries)[:10]}"
    )
    assert not missing_documents, (
        f"{language}: qrels reference {len(missing_documents)} "
        f"missing corpus documents. "
        f"Examples: {list(missing_documents)[:10]}"
    )

    data[language] = {
        "corpus_ids": corpus_ids,
        "corpus_ids_list": corpus_ids_list,
        "query_ids": query_ids,
        "query_ids_list": query_ids_list,
        "qrels": qrel_rows,
        "corpus_count": len(corpus),
        "query_count": len(queries),
        "qrel_count": len(qrels),
    }

    print(
        f"  corpus={len(corpus):,}, "
        f"queries={len(queries):,}, "
        f"qrels={len(qrels):,}"
    )


# ---------------------------------------------------------
# Compare Italian, Spanish, and Chinese
# ---------------------------------------------------------

all_checks_passed = True
baseline = "zh"

for language in ["it", "es"]:
    all_checks_passed &= show_difference(
        "Corpus ID comparison",
        data[language]["corpus_ids"],
        data[baseline]["corpus_ids"],
        language,
        baseline,
    )

    all_checks_passed &= show_difference(
        "Query ID comparison",
        data[language]["query_ids"],
        data[baseline]["query_ids"],
        language,
        baseline,
    )

    all_checks_passed &= show_difference(
        "Qrel comparison",
        data[language]["qrels"],
        data[baseline]["qrels"],
        language,
        baseline,
    )


# ---------------------------------------------------------
# Load the existing MTEB Chinese reference task
# ---------------------------------------------------------

print("\nLoading the existing MTEB reference dataset...")

reference_corpus = load_component(
    REFERENCE_REPO_ID,
    "corpus",
    "dev",
)

reference_queries = load_component(
    REFERENCE_REPO_ID,
    "queries",
    "dev",
)

reference_qrels = load_component(
    REFERENCE_REPO_ID,
    "default",
    "dev",
)

reference_corpus_id_column = get_id_column(reference_corpus)
reference_query_id_column = get_id_column(reference_queries)

reference_corpus_ids = {
    str(value)
    for value in reference_corpus[reference_corpus_id_column]
}

reference_query_ids = {
    str(value)
    for value in reference_queries[reference_query_id_column]
}

reference_qrel_rows = {
    (
        str(row["query-id"]),
        str(row["corpus-id"]),
        int(row["score"]),
    )
    for row in reference_qrels
}


# ---------------------------------------------------------
# Compare your Chinese subset against MTEB
# ---------------------------------------------------------

all_checks_passed &= show_difference(
    "Chinese corpus IDs versus existing MTEB",
    data["zh"]["corpus_ids"],
    reference_corpus_ids,
    "your zh",
    "MTEB",
)

all_checks_passed &= show_difference(
    "Chinese query IDs versus existing MTEB",
    data["zh"]["query_ids"],
    reference_query_ids,
    "your zh",
    "MTEB",
)

all_checks_passed &= show_difference(
    "Chinese qrels versus existing MTEB",
    data["zh"]["qrels"],
    reference_qrel_rows,
    "your zh",
    "MTEB",
)


# ---------------------------------------------------------
# Optional: verify identical ordering
# ---------------------------------------------------------

for language in ["it", "es"]:
    same_corpus_order = (
        data[language]["corpus_ids_list"]
        == data["zh"]["corpus_ids_list"]
    )
    same_query_order = (
        data[language]["query_ids_list"]
        == data["zh"]["query_ids_list"]
    )

    print(
        f"\n{language} vs zh ordering:"
        f"\n  Same corpus order: {same_corpus_order}"
        f"\n  Same query order: {same_query_order}"
    )


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

expected_counts = {
    "corpus": 106_813,
    "queries": 6_980,
    "qrels": 7_437,
}

for language in LANGUAGES:
    assert data[language]["corpus_count"] == expected_counts["corpus"], (
        f"{language}: unexpected corpus size"
    )
    assert data[language]["query_count"] == expected_counts["queries"], (
        f"{language}: unexpected query count"
    )
    assert data[language]["qrel_count"] == expected_counts["qrels"], (
        f"{language}: unexpected qrel count"
    )

assert all_checks_passed, (
    "At least one language differs from another language "
    "or from the MTEB reference."
)

print("\n✅ All alignment checks passed.")
print("✅ Italian, Spanish, and Chinese use identical document IDs.")
print("✅ Italian, Spanish, and Chinese use identical query IDs.")
print("✅ Italian, Spanish, and Chinese use identical qrels.")
print("✅ The Chinese subset matches the existing MTEB task.")

Loading it...


README.md:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

it-corpus/dev-00000-of-00001.parquet:   0%|          | 0.00/22.3M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/106813 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/808731 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/6980 [00:00<?, ? examples/s]

... (18 outputs left out)

Generating dev split:   0%|          | 0/6980 [00:00<?, ? examples/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/7437 [00:00<?, ? examples/s]


Chinese corpus IDs versus existing MTEB
  your zh: 106,813
  MTEB: 106,813
  Only in your zh: 0
  Only in MTEB: 0

Chinese query IDs versus existing MTEB
  your zh: 6,980
  MTEB: 6,980
  Only in your zh: 0
  Only in MTEB: 0

Chinese qrels versus existing MTEB
  your zh: 7,437
  MTEB: 7,437
  Only in your zh: 0
  Only in MTEB: 0

it vs zh ordering:
  Same corpus order: True
  Same query order: False

es vs zh ordering:
  Same corpus order: True
  Same query order: False

✅ All alignment checks passed.
✅ Italian, Spanish, and Chinese use identical document IDs.
✅ Italian, Spanish, and Chinese use identical query IDs.
✅ Italian, Spanish, and Chinese use identical qrels.
✅ The Chinese subset matches the existing MTEB task.


In [5]:
import ir_datasets
from datasets import load_dataset


# Source qrels from mMARCO v2 Chinese dev/small
source = ir_datasets.load("mmarco/v2/zh/dev/small")

ir_qrels = {
    (
        str(qrel.query_id),
        str(qrel.doc_id),
        int(qrel.relevance),
    )
    for qrel in source.qrels_iter()
}


# Existing C-MTEB qrels
cmteb_qrels_ds = load_dataset(
    "mteb/MMarcoRetrieval",
    "default",
    split="dev",
)

cmteb_qrels = {
    (
        str(row["query-id"]),
        str(row["corpus-id"]),
        int(row["score"]),
    )
    for row in cmteb_qrels_ds
}


missing_in_cmteb = ir_qrels - cmteb_qrels
extra_in_cmteb = cmteb_qrels - ir_qrels

print("ir_datasets qrels:", len(ir_qrels))
print("C-MTEB qrels:", len(cmteb_qrels))
print("Missing in C-MTEB:", len(missing_in_cmteb))
print("Extra in C-MTEB:", len(extra_in_cmteb))

if missing_in_cmteb:
    print("\nExamples missing in C-MTEB:")
    for row in list(missing_in_cmteb)[:10]:
        print(row)

if extra_in_cmteb:
    print("\nExamples extra in C-MTEB:")
    for row in list(extra_in_cmteb)[:10]:
        print(row)

assert ir_qrels == cmteb_qrels, "The qrels are not identical."

print("\n✅ C-MTEB qrels match ir_datasets exactly.")

ir_datasets qrels: 7437
C-MTEB qrels: 7437
Missing in C-MTEB: 0
Extra in C-MTEB: 0

✅ C-MTEB qrels match ir_datasets exactly.


In [6]:
from datasets import load_dataset


REPO_ID = "lopozz/MMarcoRetrieval"
LANGUAGES = ["it", "es", "zh"]


corpora = {
    lang: load_dataset(
        REPO_ID,
        f"{lang}-corpus",
        split="dev",
    )
    for lang in LANGUAGES
}

queries = {
    lang: load_dataset(
        REPO_ID,
        f"{lang}-queries",
        split="dev",
    )
    for lang in LANGUAGES
}


# Use IDs from the Italian subset.
document_id = str(corpora["it"][0]["_id"])
query_id = str(queries["it"][0]["_id"])


def find_by_id(dataset, item_id):
    matches = dataset.filter(
        lambda row: str(row["_id"]) == str(item_id)
    )

    if len(matches) == 0:
        raise KeyError(f"ID not found: {item_id}")

    return matches[0]


print("=" * 80)
print("DOCUMENT ID:", document_id)
print("=" * 80)

for lang in LANGUAGES:
    row = find_by_id(corpora[lang], document_id)

    print(f"\n[{lang}]")
    print("Title:", row.get("title", ""))
    print("Text:", row["text"])


print("\n" + "=" * 80)
print("QUERY ID:", query_id)
print("=" * 80)

for lang in LANGUAGES:
    row = find_by_id(queries[lang], query_id)

    print(f"\n[{lang}]")
    print(row["text"])

DOCUMENT ID: 3863


Filter:   0%|          | 0/106813 [00:00<?, ? examples/s]


[it]
Title: 
Text: 1 L'adulto medio ha circa 10 litri di sangue nel suo corpo. Circa 1 pinta viene data durante una donazione. 2 Un donatore sano può donare globuli rossi ogni 56 giorni o raddoppiare i globuli rossi ogni 112 giorni. Un donatore sano può donare piastrine a distanza di soli 7 giorni, ma un massimo di 24 volte l'anno.


Filter:   0%|          | 0/106813 [00:00<?, ? examples/s]


[es]
Title: 
Text: 1 El adulto promedio tiene alrededor de 10 pintas de sangre en su cuerpo. Se da aproximadamente 1 pinta durante una donación. 2 Un donante sano puede donar glóbulos rojos cada 56 días o duplicar glóbulos rojos cada 112 días. Un donante sano puede donar plaquetas con tan solo 7 días de diferencia, pero un máximo de 24 veces al año.


Filter:   0%|          | 0/106813 [00:00<?, ? examples/s]

... (3 outputs left out)

Filter:   0%|          | 0/6980 [00:00<?, ? examples/s]


[es]
que es el hermano de paula deen


Filter:   0%|          | 0/6980 [00:00<?, ? examples/s]


[zh]
宝拉·迪恩的弟弟是什么


In [7]:
from huggingface_hub import dataset_info
print(dataset_info("lopozz/MMarcoRetrieval").sha)

63ca2e67935385df799f512d84bd9c880fd579de


In [ ]:
readme = """---

language:

* it
  task_categories:
* text-retrieval
  pretty_name: Italian mMARCO v2 Retrieval
  tags:
* mteb
* retrieval
* msmarco
* mmarco
* italian
* multilingual

---

# Italian mMARCO v2 Retrieval

Italian mMARCO v2 converted to the Hugging Face retrieval format used by multilingual MTEB datasets.

The corpus is shared by the training and development queries and is stored only once.


## Citation

```bibtex
@article{Bonifacio2021MMarco,
  title={mMARCO: A Multilingual Version of MS MARCO Passage Ranking Dataset},
  author={Bonifacio, Luiz Henrique and Campiotti, Israel and Lotufo, Roberto and Nogueira, Rodrigo},
  journal={arXiv preprint arXiv:2108.13897},
  year={2021}
}
```
"""


with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

api = HfApi()

api.upload_file(
path_or_fileobj="README.md",
path_in_repo="README.md",
repo_id=REPO_ID,
repo_type="dataset",
commit_message="Add dataset card",
)

print(f"Dataset card uploaded to https://huggingface.co/datasets/{REPO_ID}")

### Train

In [2]:
dataset = ir_datasets.load("mmarco/it/train")

print(dataset.queries_count())
print(dataset.qrels_count())
print(dataset.docs_count())

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)

docs_store = dataset.docs_store()

queries_by_id = {
    query.query_id: query.text
    for query in tqdm(
        dataset.queries_iter(),
        total=dataset.queries_count(),
        desc="Loading queries",
    )
}

print("Queries loaded:", len(queries_by_id))

808731
532761
8841823


Loading queries: 100%|██████████| 808731/808731 [00:03<00:00, 211285.75it/s]

Queries loaded: 808731


In [3]:
from itertools import islice

from tqdm.auto import tqdm


BUCKETS = {
    "50-99": (50, 99, 28840),
    "100-149": (100, 149, 28840),
    "150-199": (150, 199, 15500),
    "200-299": (200, 299, 1800),
    "300-399": (300, 399, 20),
}

BATCH_SIZE = 512

selected = {
    bucket_name: []
    for bucket_name in BUCKETS
}

used_query_ids = set()

total_needed = sum(
    target
    for _, _, target in BUCKETS.values()
)


def find_bucket(length):
    for bucket_name, (minimum, maximum, _) in BUCKETS.items():
        if minimum <= length <= maximum:
            return bucket_name

    return None


def tokenize_lengths(texts):
    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )

    return [
        len(input_ids)
        for input_ids in encoded["input_ids"]
    ]


def buckets_are_full():
    return all(
        len(selected[bucket_name]) >= target
        for bucket_name, (_, _, target) in BUCKETS.items()
    )


def batched(iterator, batch_size):
    iterator = iter(iterator)

    while True:
        batch = list(islice(iterator, batch_size))

        if not batch:
            break

        yield batch

In [4]:
docpairs_iterator = dataset.docpairs_iter()

progress = tqdm(
    total=dataset.docpairs_count(),
    desc="Selecting mMARCO triples",
)

for pairs_batch in batched(docpairs_iterator, BATCH_SIZE):
    progress.update(len(pairs_batch))

    # Remove queries already selected and duplicate query IDs
    # occurring inside the current batch.
    candidate_pairs = []
    query_ids_in_batch = set()

    for pair in pairs_batch:
        if pair.query_id in used_query_ids:
            continue

        if pair.query_id in query_ids_in_batch:
            continue

        query_text = queries_by_id.get(pair.query_id)

        if not str(query_text or "").strip():
            continue

        candidate_pairs.append(pair)
        query_ids_in_batch.add(pair.query_id)

    if not candidate_pairs:
        continue

    # Load only positive documents first.
    positive_candidates = []

    for pair in candidate_pairs:
        positive_doc = docs_store.get(pair.doc_id_a)

        if positive_doc is None:
            continue

        positive_text = str(positive_doc.text or "").strip()

        if not positive_text:
            continue

        positive_candidates.append(
            {
                "pair": pair,
                "positive_text": positive_text,
            }
        )

    if not positive_candidates:
        continue

    # Tokenize all positive passages together.
    positive_lengths = tokenize_lengths(
        [
            item["positive_text"]
            for item in positive_candidates
        ]
    )

    # Keep only positives belonging to buckets that still need examples.
    accepted_candidates = []

    for item, positive_length in zip(
        positive_candidates,
        positive_lengths,
    ):
        bucket_name = find_bucket(positive_length)

        if bucket_name is None:
            continue

        _, _, target = BUCKETS[bucket_name]

        if len(selected[bucket_name]) >= target:
            continue

        accepted_candidates.append(
            {
                **item,
                "positive_length": positive_length,
                "length_bucket": bucket_name,
            }
        )

    if not accepted_candidates:
        continue

    # Load negatives only for accepted positive passages.
    complete_candidates = []

    for item in accepted_candidates:
        pair = item["pair"]

        negative_doc = docs_store.get(pair.doc_id_b)

        if negative_doc is None:
            continue

        negative_text = str(negative_doc.text or "").strip()

        if not negative_text:
            continue

        if item["positive_text"] == negative_text:
            continue

        complete_candidates.append(
            {
                **item,
                "negative_text": negative_text,
            }
        )

    if not complete_candidates:
        continue

    # Batch-tokenize negatives to record their lengths.
    negative_lengths = tokenize_lengths(
        [
            item["negative_text"]
            for item in complete_candidates
        ]
    )

    for item, negative_length in zip(
        complete_candidates,
        negative_lengths,
    ):
        pair = item["pair"]
        bucket_name = item["length_bucket"]
        positive_length = item["positive_length"]

        _, _, target = BUCKETS[bucket_name]

        # The bucket may have become full while processing this batch.
        if len(selected[bucket_name]) >= target:
            continue

        # Negative constraints are intentionally disabled for now.

        # Keep negatives inside the target context range.
        if not 50 <= negative_length <= 400:
            continue

        # Avoid a large positive/negative length mismatch.
        # length_ratio = negative_length / positive_length
        # if not 0.5 <= length_ratio <= 2.0:
        #     continue

        query_text = str(
            queries_by_id[pair.query_id]
        ).strip()

        selected[bucket_name].append(
            {
                "query_id": pair.query_id,
                "query": query_text,
                "positive_id": pair.doc_id_a,
                "positive": item["positive_text"],
                "negative_id": pair.doc_id_b,
                "negative": item["negative_text"],
                "positive_length": positive_length,
                "negative_length": negative_length,
                "length_bucket": bucket_name,
            }
        )

        used_query_ids.add(pair.query_id)

    current_total = sum(
        len(rows)
        for rows in selected.values()
    )

    progress.set_postfix(
        selected=f"{current_total:,}/{total_needed:,}",
        **{
            name: len(rows)
            for name, rows in selected.items()
        },
    )

    if buckets_are_full():
        break

progress.close()

[INFO] Opening /home/lpozzi/.ir_datasets/mmarco/italian_collection.tsv.pklz4/bin with direct file access
Selecting mMARCO triples:   7%|▋         | 2658304/39780811 [04:31<1:03:08, 9798.71it/s, 100-149=28840, 150-199=15500, 200-299=1800, 300-399=20, 50-99=28840, selected=75,000/75,000] 


In [ ]:
from pathlib import Path

from datasets import Dataset


# ---------------------------------------------------------
# 1. Convert selected buckets into one triples dataset
# ---------------------------------------------------------

rows = []

for bucket_name in BUCKETS:
    rows.extend(selected[bucket_name])

mmarco = Dataset.from_list(rows).shuffle(seed=42)

print(mmarco)


# ---------------------------------------------------------
# 2. Build MTEB-style queries, corpus, and qrels
# ---------------------------------------------------------

queries_rows = []
qrels_rows = []

# Dictionary avoids duplicating documents reused by multiple queries.
corpus_by_id = {}


for row in mmarco:
    query_id = f"mmarco-q-{row['query_id']}"
    positive_id = f"mmarco-d-{row['positive_id']}"
    negative_id = f"mmarco-d-{row['negative_id']}"

    queries_rows.append(
        {
            "_id": query_id,
            "text": row["query"],
        }
    )

    corpus_by_id[positive_id] = {
        "_id": positive_id,
        "title": "",
        "text": row["positive"],
    }

    corpus_by_id[negative_id] = {
        "_id": negative_id,
        "title": "",
        "text": row["negative"],
    }

    # Relevant document
    qrels_rows.append(
        {
            "query-id": query_id,
            "corpus-id": positive_id,
            "score": 1,
        }
    )

    # Selected hard negative
    qrels_rows.append(
        {
            "query-id": query_id,
            "corpus-id": negative_id,
            "score": 0,
        }
    )


queries_dataset = Dataset.from_list(queries_rows)

corpus_dataset = Dataset.from_list(
    list(corpus_by_id.values())
)

qrels_dataset = Dataset.from_list(qrels_rows)


print("\nQueries:")
print(queries_dataset)

print("\nCorpus:")
print(corpus_dataset)

print("\nQrels:")
print(qrels_dataset)

# ---------------------------------------------------------
# 3. Save the MTEB-style dataset
# ---------------------------------------------------------

# OUTPUT_DIR = Path("../data/mmarco_it_25k")
OUTPUT_DIR = Path("../data/mmarco_it_75k")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queries_path = OUTPUT_DIR / "train_queries.parquet"
corpus_path = OUTPUT_DIR / "train_corpus.parquet"
qrels_path = OUTPUT_DIR / "train_qrels.parquet"
triples_path = OUTPUT_DIR / "train_triples.parquet"

queries_dataset.to_parquet(str(queries_path))
corpus_dataset.to_parquet(str(corpus_path))
qrels_dataset.to_parquet(str(qrels_path))

# Keep the original triples too.
mmarco.to_parquet(str(triples_path))

print("\nSaved files:")
print(queries_path)
print(corpus_path)
print(qrels_path)
print(triples_path)

Dataset({
    features: ['query_id', 'query', 'positive_id', 'positive', 'negative_id', 'negative', 'positive_length', 'negative_length', 'length_bucket'],
    num_rows: 75000
})

Queries:
Dataset({
    features: ['_id', 'text'],
    num_rows: 75000
})

Corpus:
Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 106165
})

Qrels:
Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 150000
})


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:09<00:00,  9.56s/ba]



Saved files:
../data/mmarco_it_65k/train_queries.parquet
../data/mmarco_it_65k/train_corpus.parquet
../data/mmarco_it_65k/train_qrels.parquet
../data/mmarco_it_65k/train_triples.parquet


In [ ]:
import pandas as pd

lengths = mmarco.select_columns(
    ["positive_length", "negative_length"]
).to_pandas()

print(
    lengths.describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

from collections import Counter

positive_qrels = qrels_dataset.filter(
    lambda row: row["score"] > 0
)

relevant_docs_per_query = Counter(
    positive_qrels["query-id"]
)

distribution = Counter(
    relevant_docs_per_query.values()
)

print("Relevant-document distribution per query:")

for n_relevant, n_queries in sorted(distribution.items()):
    print(
        f"{n_queries:,} queries have "
        f"{n_relevant} relevant document(s)"
    )

       positive_length  negative_length
count     25000.000000     25000.000000
mean        126.790080        87.491440
std          47.655152        31.772946
min          50.000000        50.000000
10%          64.000000        56.000000
25%          84.000000        64.000000
50%         124.000000        76.000000
75%         160.000000       104.000000
90%         188.000000       136.000000
95%         207.000000       151.000000
99%         243.000000       181.000000
max         395.000000       372.000000


Filter: 100%|██████████| 50000/50000 [00:00<00:00, 666295.58 examples/s]


Relevant-document distribution per query:
25,000 queries have 1 relevant document(s)


### Test

In [1]:
from pathlib import Path
import random

import ir_datasets
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

DATASET_ID = "mmarco/it/dev/small"

MIN_TOKENS = 1 # 50
MAX_TOKENS = 510 # 400

N_DISTRACTORS = 50_000
# N_DISTRACTORS = 10_000
RANDOM_SEED = 42

OUTPUT_DIR = Path(
    "../data/mmarco_it_dev_small_50k_len1_500"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Load dataset and tokenizer
# ---------------------------------------------------------

dataset = ir_datasets.load(DATASET_ID)

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)

# Prevent warnings while counting tokens without truncation.
tokenizer.model_max_length = 10_000_000

docs_store = dataset.docs_store()

print("Original documents:", dataset.docs_count())
print("Original queries:", dataset.queries_count())
print("Original qrels:", dataset.qrels_count())


def token_length(text: str) -> int:
    return len(
        tokenizer(
            str(text or ""),
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
    )


# ---------------------------------------------------------
# Load queries
# ---------------------------------------------------------

query_text_by_id = {
    str(query.query_id): str(query.text or "").strip()
    for query in tqdm(
        dataset.queries_iter(),
        total=dataset.queries_count(),
        desc="Loading queries",
    )
}


# ---------------------------------------------------------
# Keep only positive qrels whose relevant document is
# between MIN_TOKENS and MAX_TOKENS tokens
# ---------------------------------------------------------

qrels_rows = []
relevant_documents = {}

for qrel in tqdm(
    dataset.qrels_iter(),
    total=dataset.qrels_count(),
    desc="Filtering relevant documents",
):
    if qrel.relevance <= 0:
        continue

    query_id = str(qrel.query_id)
    doc_id = str(qrel.doc_id)

    doc = docs_store.get(doc_id)

    if doc is None:
        continue

    text = str(doc.text or "").strip()

    if not text:
        continue

    length = token_length(text)

    if not MIN_TOKENS <= length <= MAX_TOKENS:
        continue

    relevant_documents[doc_id] = {
        "_id": doc_id,
        "title": "",
        "text": text,
        "raw_token_length": length,
    }

    qrels_rows.append(
        {
            "query-id": query_id,
            "corpus-id": doc_id,
            "score": int(qrel.relevance),
        }
    )


qrels_dataset = Dataset.from_list(qrels_rows)

valid_query_ids = set(qrels_dataset["query-id"])
relevant_doc_ids = set(relevant_documents)

print("Queries with an eligible relevant document:", len(valid_query_ids))
print("Eligible relevant documents:", len(relevant_doc_ids))
print("Eligible qrels:", len(qrels_dataset))


# ---------------------------------------------------------
# Build the filtered query dataset
# ---------------------------------------------------------

queries_rows = [
    {
        "_id": query_id,
        "text": query_text_by_id[query_id],
    }
    for query_id in sorted(valid_query_ids)
    if query_id in query_text_by_id
]

queries_dataset = Dataset.from_list(queries_rows)


# ---------------------------------------------------------
# Reservoir-sample distractors between 50 and 400 tokens
# ---------------------------------------------------------

random_generator = random.Random(RANDOM_SEED)

selected_distractors = []
seen_eligible_distractors = 0

for doc in tqdm(
    dataset.docs_iter(),
    total=dataset.docs_count(),
    desc="Sampling length-filtered distractors",
):
    doc_id = str(doc.doc_id)

    if doc_id in relevant_doc_ids:
        continue

    text = str(doc.text or "").strip()

    if not text:
        continue

    length = token_length(text)

    if not MIN_TOKENS <= length <= MAX_TOKENS:
        continue

    seen_eligible_distractors += 1

    row = {
        "_id": doc_id,
        "title": "",
        "text": text,
        "raw_token_length": length,
    }

    if len(selected_distractors) < N_DISTRACTORS:
        selected_distractors.append(row)
    else:
        replacement_index = random_generator.randrange(
            seen_eligible_distractors
        )

        if replacement_index < N_DISTRACTORS:
            selected_distractors[replacement_index] = row


if len(selected_distractors) < N_DISTRACTORS:
    raise RuntimeError(
        f"Only {len(selected_distractors):,} eligible distractors "
        f"were found, but {N_DISTRACTORS:,} were requested."
    )


# ---------------------------------------------------------
# Build corpus
# ---------------------------------------------------------

corpus_rows = (
    list(relevant_documents.values())
    + selected_distractors
)

corpus_dataset = Dataset.from_list(corpus_rows)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

query_ids = set(queries_dataset["_id"])
corpus_ids = set(corpus_dataset["_id"])

missing_query_ids = {
    query_id
    for query_id in qrels_dataset["query-id"]
    if query_id not in query_ids
}

missing_document_ids = {
    document_id
    for document_id in qrels_dataset["corpus-id"]
    if document_id not in corpus_ids
}

assert not missing_query_ids
assert not missing_document_ids
assert relevant_doc_ids.issubset(corpus_ids)

assert min(corpus_dataset["raw_token_length"]) >= MIN_TOKENS
assert max(corpus_dataset["raw_token_length"]) <= MAX_TOKENS


# ---------------------------------------------------------
# Save MTEB-style files
# ---------------------------------------------------------

queries_path = OUTPUT_DIR / "test_queries.parquet"
corpus_path = OUTPUT_DIR / "test_corpus.parquet"
qrels_path = OUTPUT_DIR / "test_qrels.parquet"

queries_dataset.to_parquet(str(queries_path))

# Remove analysis metadata for a clean MTEB corpus.
corpus_dataset.remove_columns(
    "raw_token_length"
).to_parquet(
    str(corpus_path)
)

qrels_dataset.to_parquet(str(qrels_path))


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\nConversion complete")
print("Queries:", len(queries_dataset))
print("Corpus documents:", len(corpus_dataset))
print("Positive qrels:", len(qrels_dataset))
print("Unique relevant documents:", len(relevant_doc_ids))
print("Sampled distractors:", len(selected_distractors))

print(
    "Document length range:",
    min(corpus_dataset["raw_token_length"]),
    "-",
    max(corpus_dataset["raw_token_length"]),
)

print("\nSaved:")
print(queries_path)
print(corpus_path)
print(qrels_path)

Original documents: 8841823
Original queries: 6980
Original qrels: 7437


Loading queries:   0%|          | 0/6980 [00:00<?, ?it/s]

Filtering relevant documents:   0%|          | 0/7437 [00:00<?, ?it/s]

[INFO] Opening /home/lpozzi/.ir_datasets/mmarco/italian_collection.tsv.pklz4/bin with direct file access


Queries with an eligible relevant document: 6980
Eligible relevant documents: 7433
Eligible qrels: 7437


Sampling length-filtered distractors:   0%|          | 0/8841823 [00:00<?, ?it/s]

[INFO] Opening /home/lpozzi/.ir_datasets/mmarco/italian_collection.tsv.pklz4/bin with direct file access


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


Conversion complete
Queries: 6980
Corpus documents: 57433
Positive qrels: 7437
Unique relevant documents: 7433
Sampled distractors: 50000
Document length range: 6 - 453

Saved:
../data/mmarco_it_dev_small_50k_len1_500/test_queries.parquet
../data/mmarco_it_dev_small_50k_len1_500/test_corpus.parquet
../data/mmarco_it_dev_small_50k_len1_500/test_qrels.parquet


In [17]:
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)


# ---------------------------------------------------------
# 1. Add token lengths to the entire test corpus
# ---------------------------------------------------------

def add_token_lengths(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )

    return {
        "raw_token_length": [
            len(input_ids)
            for input_ids in encoded["input_ids"]
        ]
    }


corpus_with_lengths = corpus_dataset.map(
    add_token_lengths,
    batched=True,
    batch_size=512,
    desc="Computing test corpus lengths",
)


# ---------------------------------------------------------
# 2. Summary statistics for the full test corpus
# ---------------------------------------------------------

corpus_lengths = pd.Series(
    corpus_with_lengths["raw_token_length"],
    name="raw_token_length",
)

print("Full test corpus length statistics:")
print(
    corpus_lengths.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ---------------------------------------------------------
# 3. Distribution in fixed length buckets
# ---------------------------------------------------------

bins = [
    0,
    49,
    99,
    149,
    199,
    299,
    400,
    511,
    1_000,
    float("inf"),
]

labels = [
    "0-49",
    "50-99",
    "100-149",
    "150-199",
    "200-299",
    "300-400",
    "401-511",
    "512-1000",
    ">1000",
]

corpus_length_buckets = pd.cut(
    corpus_lengths,
    bins=bins,
    labels=labels,
    include_lowest=True,
)

corpus_distribution = (
    corpus_length_buckets
    .value_counts()
    .reindex(labels, fill_value=0)
    .to_frame("documents")
)

corpus_distribution["share"] = (
    corpus_distribution["documents"]
    / len(corpus_with_lengths)
)

print("\nFull test corpus distribution:")
print(corpus_distribution)


# ---------------------------------------------------------
# 4. Relevant documents per query
# ---------------------------------------------------------

positive_qrels = qrels_dataset.filter(
    lambda row: row["score"] > 0
)

relevant_docs_per_query = Counter(
    positive_qrels["query-id"]
)

relevance_distribution = Counter(
    relevant_docs_per_query.values()
)

print("\nRelevant-document distribution per query:")

for n_relevant, n_queries in sorted(
    relevance_distribution.items()
):
    print(
        f"{n_queries:,} queries have "
        f"{n_relevant} relevant document(s)"
    )


# Check queries without positive qrels
all_query_ids = set(queries_dataset["_id"])
queries_with_positive = set(relevant_docs_per_query)

print(
    "Queries without a positive qrel:",
    len(all_query_ids - queries_with_positive),
)


# ---------------------------------------------------------
# 5. Length statistics for relevant documents only
# ---------------------------------------------------------

length_by_doc_id = {
    doc_id: length
    for doc_id, length in zip(
        corpus_with_lengths["_id"],
        corpus_with_lengths["raw_token_length"],
    )
}

relevant_lengths = [
    length_by_doc_id[doc_id]
    for doc_id in positive_qrels["corpus-id"]
]

relevant_lengths = pd.Series(
    relevant_lengths,
    name="relevant_document_length",
)

print("\nRelevant-document length statistics:")
print(
    relevant_lengths.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ---------------------------------------------------------
# 6. Relevant-document distribution by length bucket
# ---------------------------------------------------------

relevant_length_buckets = pd.cut(
    relevant_lengths,
    bins=bins,
    labels=labels,
    include_lowest=True,
)

relevant_distribution = (
    relevant_length_buckets
    .value_counts()
    .reindex(labels, fill_value=0)
    .to_frame("relevant_documents")
)

relevant_distribution["share"] = (
    relevant_distribution["relevant_documents"]
    / len(relevant_lengths)
)

print("\nRelevant-document length distribution:")
print(relevant_distribution)

Computing test corpus lengths: 100%|██████████| 56569/56569 [00:01<00:00, 39638.02 examples/s]


Full test corpus length statistics:
count    56569.000000
mean        83.910057
std         29.932866
min         50.000000
10%         56.000000
25%         63.000000
50%         74.000000
75%         97.000000
90%        128.000000
95%        144.000000
99%        181.000000
max        394.000000
Name: raw_token_length, dtype: float64

Full test corpus distribution:
                  documents     share
raw_token_length                     
0-49                      0  0.000000
50-99                 43328  0.765932
100-149               10982  0.194135
150-199                1997  0.035302
200-299                 254  0.004490
300-400                   8  0.000141
401-511                   0  0.000000
512-1000                  0  0.000000
>1000                     0  0.000000


Filter: 100%|██████████| 6571/6571 [00:00<00:00, 547209.86 examples/s]


Relevant-document distribution per query:
5,863 queries have 1 relevant document(s)
275 queries have 2 relevant document(s)
42 queries have 3 relevant document(s)
8 queries have 4 relevant document(s)
Queries without a positive qrel: 0



Relevant-document length statistics:
count    6571.000000
mean       83.667935
std        28.649691
min        50.000000
10%        56.000000
25%        64.000000
50%        75.000000
75%        95.000000
90%       124.000000
95%       141.500000
99%       181.000000
max       378.000000
Name: relevant_document_length, dtype: float64

Relevant-document length distribution:
                          relevant_documents     share
relevant_document_length                              
0-49                                       0  0.000000
50-99                                   5126  0.780094
100-149                                 1202  0.182925
150-199                                  215  0.032720
200-299                                   27  0.004109
300-400                                    1  0.000152
401-511                                    0  0.000000
512-1000                                   0  0.000000
>1000                                      0  0.000000


In [19]:
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer


DATA_DIR = Path("../data/mmarco_it_dev_small_50k_len50_400").resolve()
SPLIT = "test"

corpus = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_corpus.parquet")
)

queries = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_queries.parquet")
)

qrels = Dataset.from_parquet(
    str(DATA_DIR / f"{SPLIT}_qrels.parquet")
)

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)


corpus_with_lengths = corpus.map(
    add_length,
    desc="Computing document lengths",
)

corpus_id_column = find_column(
    corpus_with_lengths,
    ("_id", "id", "corpus-id", "corpus_id"),
)

query_id_column = find_column(
    queries,
    ("_id", "id", "query-id", "query_id"),
)

qrel_query_column = find_column(
    qrels,
    ("query-id", "query_id", "query-id"),
)

qrel_corpus_column = find_column(
    qrels,
    ("corpus-id", "corpus_id", "corpus-id"),
)


corpus_by_id = {
    str(row[corpus_id_column]): row
    for row in corpus_with_lengths
}

queries_by_id = {
    str(row[query_id_column]): row
    for row in queries
}

Computing document lengths: 100%|██████████| 56569/56569 [00:11<00:00, 5142.51 examples/s]


In [ ]:
inspect_pairs_by_document_length(50, 99) # 5126
# inspect_pairs_by_document_length(67, 338) # buco
# inspect_pairs_by_document_length(339, 450) # 110
# inspect_pairs_by_document_length(450, 500) # 315
# inspect_pairs_by_document_length(500, 1000) # 616

5126 relevant query-document pairs with documents between 50 and 99 tokens
Query ID: 1090377
Document ID: 8001367
Document length: 73
Relevance score: 1
Query: Costo della cabina MBA
Title: 
Document: 'Costo del programma MBA. Il costo completo del programma (tuizione e tasse) sia per i residenti che per i non residenti alla University of Chicago Booth School of Business è di $127.960 USD. Questo è uno dei programmi più costosi rispetto al costo medio residente in tutti i programmi MBA ($26.785 dollari degli Stati Uniti).'
Query ID: 1087544
Document ID: 7101182
Document length: 70
Relevance score: 1
Query: Di cosa sono responsabili gli investigatori e gli agenti speciali?
Title: 
Document: 'In realtà, decine di agenzie federali hanno la loro autorità di contrasto, e agenti speciali che utilizzano tale autorità ogni giorno. La sicurezza sociale non è diversa. Abbiamo investigatori criminali, agenti speciali, che sono responsabili di indagare sulle violazioni delle leggi che governano i 

## istella22
https://iris.cnr.it/retrieve/b01c53e5-6005-4b5f-a706-69c461a657cd/prod_469573-doc_190234.pdf

### Test

In [5]:
from pathlib import Path

import ir_datasets
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

DATASET_ID = "istella22/test"

MIN_TOKENS = 1
MAX_TOKENS = 7_000_000

OUTPUT_DIR = Path("../data/istella22_test_full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Load dataset and tokenizer
# ---------------------------------------------------------


dataset = ir_datasets.load(DATASET_ID)

print("Dataset ID:", DATASET_ID)
print(dataset)
print("Queries handler:", type(dataset.queries_handler()))

tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)

tokenizer.model_max_length = 10_000_000

print("Original documents:", dataset.docs_count())
print("Original queries:", dataset.queries_count())
print("Original qrels:", dataset.qrels_count())


def token_length(text: str) -> int:
    return len(
        tokenizer(
            str(text or ""),
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
    )


# ---------------------------------------------------------
# Load every query
# ---------------------------------------------------------

queries_rows = [
    {
        "_id": str(query.query_id),
        "text": str(query.text or "").strip(),
    }
    for query in tqdm(
        dataset.queries_iter(),
        total=dataset.queries_count(),
        desc="Loading queries",
    )
    if str(query.text or "").strip()
]

queries_dataset = Dataset.from_list(queries_rows)
query_ids = set(queries_dataset["_id"])


# ---------------------------------------------------------
# Load every eligible corpus document
# ---------------------------------------------------------

corpus_rows = []

for doc in tqdm(
    dataset.docs_iter(),
    total=dataset.docs_count(),
    desc="Loading complete corpus",
):
    doc_id = str(doc.doc_id)
    text = str(doc.text or "").strip()

    if not text:
        continue

    length = token_length(text)

    if not MIN_TOKENS <= length <= MAX_TOKENS:
        continue

    corpus_rows.append(
        {
            "_id": doc_id,
            "title": "",
            "text": text,
            "raw_token_length": length,
        }
    )

corpus_dataset = Dataset.from_list(corpus_rows)
corpus_ids = set(corpus_dataset["_id"])


# ---------------------------------------------------------
# Keep positive qrels whose query and document are present
# ---------------------------------------------------------

qrels_rows = []

for qrel in tqdm(
    dataset.qrels_iter(),
    total=dataset.qrels_count(),
    desc="Loading positive qrels",
):
    if qrel.relevance <= 0:
        continue

    query_id = str(qrel.query_id)
    doc_id = str(qrel.doc_id)

    if query_id not in query_ids:
        continue

    if doc_id not in corpus_ids:
        continue

    qrels_rows.append(
        {
            "query-id": query_id,
            "corpus-id": doc_id,
            "score": int(qrel.relevance),
        }
    )

qrels_dataset = Dataset.from_list(qrels_rows)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

missing_query_ids = set(qrels_dataset["query-id"]) - query_ids
missing_document_ids = set(qrels_dataset["corpus-id"]) - corpus_ids

assert not missing_query_ids
assert not missing_document_ids

if len(corpus_dataset) > 0:
    assert min(corpus_dataset["raw_token_length"]) >= MIN_TOKENS
    assert max(corpus_dataset["raw_token_length"]) <= MAX_TOKENS


# ---------------------------------------------------------
# Save MTEB-style files
# ---------------------------------------------------------

queries_path = OUTPUT_DIR / "test_queries.parquet"
corpus_path = OUTPUT_DIR / "test_corpus.parquet"
qrels_path = OUTPUT_DIR / "test_qrels.parquet"

queries_dataset.to_parquet(str(queries_path))

corpus_dataset.remove_columns(
    "raw_token_length"
).to_parquet(str(corpus_path))

qrels_dataset.to_parquet(str(qrels_path))


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\nConversion complete")
print("Queries:", len(queries_dataset))
print("Corpus documents:", len(corpus_dataset))
print("Positive qrels:", len(qrels_dataset))

if len(corpus_dataset) > 0:
    print(
        "Document length range:",
        min(corpus_dataset["raw_token_length"]),
        "-",
        max(corpus_dataset["raw_token_length"]),
    )

print("\nSaved:")
print(queries_path)
print(corpus_path)
print(qrels_path)

Dataset ID: istella22/test
Dataset(id='istella22/test', provides=['docs', 'queries', 'qrels'])
Queries handler: <class 'ir_datasets.formats.jsonl.JsonlQueries'>
Original documents: 8421456
Original queries: 2198
Original qrels: 10693


[INFO] If you have a local copy of https://www.istella.ai/dataset/istella22.tar.gz, you can symlink it here to avoid downloading it again: /home/lpozzi/.ir_datasets/downloads/c2e49dca9730fbb14164ed890756dc1d
[INFO] [starting] https://www.istella.ai/dataset/istella22.tar.gz
                                                         
                                                                          [INFO] [finished] https://www.istella.ai/dataset/istella22.tar.gz: [00:00] [27.3kB] [46.7MB/s]
[WARNING] Download failed: Expected md5 hash to be c2e49dca9730fbb14164ed890756dc1d but got 6a7cf1a670e13ca61e0d3b969928b097
Loading queries:   0%|          | 0/2198 [00:00<?, ?it/s]


HashVerificationError: Expected md5 hash to be c2e49dca9730fbb14164ed890756dc1d but got 6a7cf1a670e13ca61e0d3b969928b097

In [6]:
import itertools
import ir_datasets

dataset = ir_datasets.load("istella22/test")

print("Queries:", dataset.queries_count())
print("First query:", next(iter(dataset.queries_iter())))
print("First qrel:", next(iter(dataset.qrels_iter())))
print("First document:", next(iter(dataset.docs_iter())))

Queries: 2198


[INFO] If you have a local copy of https://www.istella.ai/dataset/istella22.tar.gz, you can symlink it here to avoid downloading it again: /home/lpozzi/.ir_datasets/downloads/c2e49dca9730fbb14164ed890756dc1d
[INFO] [starting] https://www.istella.ai/dataset/istella22.tar.gz
[INFO] [finished] https://www.istella.ai/dataset/istella22.tar.gz: [00:00] [27.3kB] [54.0MB/s]
[WARNING] Download failed: Expected md5 hash to be c2e49dca9730fbb14164ed890756dc1d but got 6a7cf1a670e13ca61e0d3b969928b097


HashVerificationError: Expected md5 hash to be c2e49dca9730fbb14164ed890756dc1d but got 6a7cf1a670e13ca61e0d3b969928b097

In [4]:
import ir_datasets

ir_home = ir_datasets.util.home_path()
print(ir_home)
print(ir_home / "istella22")


from pathlib import Path

istella_dir = Path(ir_home) / "istella22"

for path in istella_dir.rglob("*"):
    if path.is_file():
        print(path, path.stat().st_size)

import shutil
from pathlib import Path
import ir_datasets

istella_dir = Path(ir_datasets.util.home_path()) / "istella22"

print("Removing:", istella_dir)
shutil.rmtree(istella_dir, ignore_errors=True)

/home/lpozzi/.ir_datasets
/home/lpozzi/.ir_datasets/istella22
Removing: /home/lpozzi/.ir_datasets/istella22


In [ ]:
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(
    "nickprock/splade-bert-base-italian-xxl-uncased-cv"
)


# ---------------------------------------------------------
# 1. Add token lengths to the entire test corpus
# ---------------------------------------------------------

def add_token_lengths(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )

    return {
        "raw_token_length": [
            len(input_ids)
            for input_ids in encoded["input_ids"]
        ]
    }


corpus_with_lengths = corpus_dataset.map(
    add_token_lengths,
    batched=True,
    batch_size=512,
    desc="Computing test corpus lengths",
)


# ---------------------------------------------------------
# 2. Summary statistics for the full test corpus
# ---------------------------------------------------------

corpus_lengths = pd.Series(
    corpus_with_lengths["raw_token_length"],
    name="raw_token_length",
)

print("Full test corpus length statistics:")
print(
    corpus_lengths.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ---------------------------------------------------------
# 3. Distribution in fixed length buckets
# ---------------------------------------------------------

bins = [
    0,
    49,
    99,
    149,
    199,
    299,
    400,
    511,
    1_000,
    float("inf"),
]

labels = [
    "0-49",
    "50-99",
    "100-149",
    "150-199",
    "200-299",
    "300-400",
    "401-511",
    "512-1000",
    ">1000",
]

corpus_length_buckets = pd.cut(
    corpus_lengths,
    bins=bins,
    labels=labels,
    include_lowest=True,
)

corpus_distribution = (
    corpus_length_buckets
    .value_counts()
    .reindex(labels, fill_value=0)
    .to_frame("documents")
)

corpus_distribution["share"] = (
    corpus_distribution["documents"]
    / len(corpus_with_lengths)
)

print("\nFull test corpus distribution:")
print(corpus_distribution)


# ---------------------------------------------------------
# 4. Relevant documents per query
# ---------------------------------------------------------

positive_qrels = qrels_dataset.filter(
    lambda row: row["score"] > 0
)

relevant_docs_per_query = Counter(
    positive_qrels["query-id"]
)

relevance_distribution = Counter(
    relevant_docs_per_query.values()
)

print("\nRelevant-document distribution per query:")

for n_relevant, n_queries in sorted(
    relevance_distribution.items()
):
    print(
        f"{n_queries:,} queries have "
        f"{n_relevant} relevant document(s)"
    )


# Check queries without positive qrels
all_query_ids = set(queries_dataset["_id"])
queries_with_positive = set(relevant_docs_per_query)

print(
    "Queries without a positive qrel:",
    len(all_query_ids - queries_with_positive),
)


# ---------------------------------------------------------
# 5. Length statistics for relevant documents only
# ---------------------------------------------------------

length_by_doc_id = {
    doc_id: length
    for doc_id, length in zip(
        corpus_with_lengths["_id"],
        corpus_with_lengths["raw_token_length"],
    )
}

relevant_lengths = [
    length_by_doc_id[doc_id]
    for doc_id in positive_qrels["corpus-id"]
]

relevant_lengths = pd.Series(
    relevant_lengths,
    name="relevant_document_length",
)

print("\nRelevant-document length statistics:")
print(
    relevant_lengths.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ---------------------------------------------------------
# 6. Relevant-document distribution by length bucket
# ---------------------------------------------------------

relevant_length_buckets = pd.cut(
    relevant_lengths,
    bins=bins,
    labels=labels,
    include_lowest=True,
)

relevant_distribution = (
    relevant_length_buckets
    .value_counts()
    .reindex(labels, fill_value=0)
    .to_frame("relevant_documents")
)

relevant_distribution["share"] = (
    relevant_distribution["relevant_documents"]
    / len(relevant_lengths)
)

print("\nRelevant-document length distribution:")
print(relevant_distribution)

## sberquad

In [11]:
from transformers import AutoTokenizer
import requests
model_name = "google/translategemma-12b-it"
tok = AutoTokenizer.from_pretrained(model_name)
VLLM_API_KEY="0cb0d80524dd713485412a923c51798295b01eca66ce8d0a3069377fbab10d1a"

messages = [{
    "role": "user",
    "content": [{
        "type": "text",
        "source_lang_code": "ru",
        "target_lang_code": "it",
        "text": "Чем связаны между собой листы макромолекул?",
        "image": None,
    }],
}]

prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

response = requests.post(
    "http://localhost:8000/v1/completions",
    headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
    json={"model": model_name, "prompt": prompt, "temperature": 0, "max_tokens": 64},
)

response.raise_for_status()

translation = response.json()["choices"][0]['text']
print(translation)

Come sono collegati tra loro i fogli di macromolecole?


In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset


DATASET_NAME = "kuznetsoffandrey/sberquad"
OUTPUT_DIR = Path("../data/sberquad_mteb").resolve()


def stable_context_id(context: str) -> str:
    """
    Generate a deterministic ID from the complete context.

    SHA-1 is used only as a stable identifier, not for security.
    """
    digest = hashlib.sha1(
        context.strip().encode("utf-8")
    ).hexdigest()

    return f"doc-{digest}"


def convert_split(
    dataset: Dataset,
) -> tuple[Dataset, Dataset, Dataset]:
    """
    Convert one SberQuAD split into queries, corpus and qrels.

    Output schemas:

    queries:
        query-id, query

    corpus:
        corpus-id, text, title

    qrels:
        query-id, corpus-id, score
    """
    query_rows = []
    corpus_by_id = {}
    qrel_rows = []

    for _, row in enumerate(dataset):
        context = row["context"].strip()
        question = row["question"].strip()

        if not context or not question:
            continue

        query_id = f"query-{row['id']}"
        corpus_id = stable_context_id(context)

        query_rows.append(
            {
                "_id": query_id,
                "text": question,
            }
        )

        # Multiple questions can refer to the same context.
        if corpus_id not in corpus_by_id:
            corpus_by_id[corpus_id] = {
                "_id": corpus_id,
                "text": context,
                "title": row.get("title") or "",
            }

        qrel_rows.append(
            {
                "query-id": query_id,
                "corpus-id": corpus_id,
                "score": 1,
            }
        )

    queries = Dataset.from_list(query_rows)
    corpus = Dataset.from_list(list(corpus_by_id.values()))
    qrels = Dataset.from_list(qrel_rows)

    return queries, corpus, qrels


raw_dataset = load_dataset(DATASET_NAME)

queries_splits = {}
corpus_splits = {}
qrels_splits = {}

for split_name, split_dataset in raw_dataset.items():
    print(f"Converting {split_name}: {len(split_dataset):,} rows")

    split_queries, split_corpus, split_qrels = convert_split(
        split_dataset
    )

    queries_splits[split_name] = split_queries
    corpus_splits[split_name] = split_corpus
    qrels_splits[split_name] = split_qrels

    print(
        f"  Queries: {len(split_queries):,}\n"
        f"  Unique documents: {len(split_corpus):,}\n"
        f"  Qrels: {len(split_qrels):,}"
    )


queries = DatasetDict(queries_splits)
corpus = DatasetDict(corpus_splits)
qrels = DatasetDict(qrels_splits)

print(queries)
print(corpus)
print(qrels)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name in queries:
    queries[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_queries.parquet"
    )

    corpus[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_corpus.parquet"
    )

    qrels[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_qrels.parquet"
    )

print(f"Saved converted dataset to: {OUTPUT_DIR}")

Converting train: 45,328 rows
  Queries: 45,328
  Unique documents: 9,078
  Qrels: 45,328
Converting validation: 5,036 rows
  Queries: 5,036
  Unique documents: 3,971
  Qrels: 5,036
Converting test: 23,936 rows
  Queries: 23,936
  Unique documents: 4,425
  Qrels: 23,936
DatasetDict({
    train: Dataset({
        features: ['_id', 'text'],
        num_rows: 45328
    })
    validation: Dataset({
        features: ['_id', 'text'],
        num_rows: 5036
    })
    test: Dataset({
        features: ['_id', 'text'],
        num_rows: 23936
    })
})
DatasetDict({
    train: Dataset({
        features: ['_id', 'text', 'title'],
        num_rows: 9078
    })
    validation: Dataset({
        features: ['_id', 'text', 'title'],
        num_rows: 3971
    })
    test: Dataset({
        features: ['_id', 'text', 'title'],
        num_rows: 4425
    })
})
DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 45328
    })
    validation: Datase

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved converted dataset to: /home/lpozzi/Git/lm-toolkit/data/sberquad_mteb


### Push to Hub

In [14]:
from datasets import Dataset

TARGET_REPO_ID = "lopozz/SberQuADRetrieval"
RU_DIR = Path("../data/sberquad_mteb").resolve()
IT_DIR = Path("../data/sberquad_mteb_itquad").resolve()

splits = sorted({path.stem.rsplit("_queries", 1)[0] for path in RU_DIR.glob("*_queries.parquet")})
print(f"Discovered splits: {splits}")

for split in splits:
    for lang, data_dir in (("ru", RU_DIR), ("it", IT_DIR)):
        for subset in ("corpus", "queries", "qrels"):
            ds = Dataset.from_parquet(str(data_dir / f"{split}_{subset}.parquet"))
            ds.push_to_hub(TARGET_REPO_ID, config_name=f"{lang}-{subset}", split=split)

print(f"Pushed to: https://huggingface.co/datasets/{TARGET_REPO_ID}")


Discovered splits: ['test', 'train', 'validation']


Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (82 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to: https://huggingface.co/datasets/lopozz/SberQuADRetrieval


## JaQuAD

In [3]:
from transformers import AutoTokenizer
import requests
model_name = "google/translategemma-12b-it"
tok = AutoTokenizer.from_pretrained(model_name)
VLLM_API_KEY="0cb0d80524dd713485412a923c51798295b01eca66ce8d0a3069377fbab10d1a"

messages = [{
    "role": "user",
    "content": [{
        "type": "text",
        "source_lang_code": "ja",
        "target_lang_code": "it",
        "text": "戦後日本のストーリー漫画の第一人者で、医学博士の一面もある漫画家は誰?",
        "image": None,
    }],
}]

prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

response = requests.post(
    "http://localhost:8000/v1/completions",
    headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
    json={"model": model_name, "prompt": prompt, "temperature": 0, "max_tokens": 64},
)

response.raise_for_status()

translation = response.json()["choices"][0]['text']
print(translation)

Qual è il famoso fumettista giapponese, attivo nel campo dei manga narrativi del dopoguerra, che è anche un medico?


In [2]:
from __future__ import annotations

import hashlib
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset


DATASET_NAME = "SkelterLabsInc/JaQuAD"
OUTPUT_DIR = Path("../data/jaquad_mteb").resolve()


def stable_context_id(context: str) -> str:
    """
    Generate a deterministic ID from the complete context.

    SHA-1 is used only as a stable identifier, not for security.
    """
    digest = hashlib.sha1(
        context.strip().encode("utf-8")
    ).hexdigest()

    return f"doc-{digest}"


def convert_split(
    dataset: Dataset,
) -> tuple[Dataset, Dataset, Dataset]:
    """
    Convert one SberQuAD split into queries, corpus and qrels.

    Output schemas:

    queries:
        query-id, query

    corpus:
        corpus-id, text, title

    qrels:
        query-id, corpus-id, score
    """
    query_rows = []
    corpus_by_id = {}
    qrel_rows = []

    for _, row in enumerate(dataset):
        context = row["context"].strip()
        question = row["question"].strip()

        if not context or not question:
            continue

        query_id = f"query-{row['id']}"
        corpus_id = stable_context_id(context)

        query_rows.append(
            {
                "_id": query_id,
                "text": question,
            }
        )

        # Multiple questions can refer to the same context.
        if corpus_id not in corpus_by_id:
            corpus_by_id[corpus_id] = {
                "_id": corpus_id,
                "text": context,
                "title": row.get("title") or "",
            }

        qrel_rows.append(
            {
                "query-id": query_id,
                "corpus-id": corpus_id,
                "score": 1,
            }
        )

    queries = Dataset.from_list(query_rows)
    corpus = Dataset.from_list(list(corpus_by_id.values()))
    qrels = Dataset.from_list(qrel_rows)

    return queries, corpus, qrels


PARQUET_REVISION = "9f859f2a319919b9bdd98b74569ad4e52e93a1e8"

data_files = {
    "train": (
        f"hf://datasets/{DATASET_NAME}@{PARQUET_REVISION}"
        "/default/train/0000.parquet"
    ),
    "validation": (
        f"hf://datasets/{DATASET_NAME}@{PARQUET_REVISION}"
        "/default/validation/0000.parquet"
    ),
}

raw_dataset = load_dataset(
    "parquet",
    data_files=data_files,
)

queries_splits = {}
corpus_splits = {}
qrels_splits = {}

for split_name, split_dataset in raw_dataset.items():
    print(f"Converting {split_name}: {len(split_dataset):,} rows")

    split_queries, split_corpus, split_qrels = convert_split(
        split_dataset
    )

    queries_splits[split_name] = split_queries
    corpus_splits[split_name] = split_corpus
    qrels_splits[split_name] = split_qrels

    print(
        f"  Queries: {len(split_queries):,}\n"
        f"  Unique documents: {len(split_corpus):,}\n"
        f"  Qrels: {len(split_qrels):,}"
    )


queries = DatasetDict(queries_splits)
corpus = DatasetDict(corpus_splits)
qrels = DatasetDict(qrels_splits)

print(queries)
print(corpus)
print(qrels)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name in queries:
    queries[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_queries.parquet"
    )

    corpus[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_corpus.parquet"
    )

    qrels[split_name].to_parquet(
        OUTPUT_DIR / f"{split_name}_qrels.parquet"
    )

print(f"Saved converted dataset to: {OUTPUT_DIR}")

default/train/0000.parquet:   0%|          | 0.00/8.53M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Converting train: 31,748 rows
  Queries: 31,748
  Unique documents: 9,713
  Qrels: 31,748
Converting validation: 3,939 rows
  Queries: 3,939
  Unique documents: 1,431
  Qrels: 3,939
DatasetDict({
    train: Dataset({
        features: ['_id', 'text'],
        num_rows: 31748
    })
    validation: Dataset({
        features: ['_id', 'text'],
        num_rows: 3939
    })
})
DatasetDict({
    train: Dataset({
        features: ['_id', 'text', 'title'],
        num_rows: 9713
    })
    validation: Dataset({
        features: ['_id', 'text', 'title'],
        num_rows: 1431
    })
})
DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 31748
    })
    validation: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 3939
    })
})


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

... (2 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved converted dataset to: /home/lpozzi/Git/lm-toolkit/data/jaquad_mteb


### Push to Hub

In [15]:
from datasets import Dataset

TARGET_REPO_ID = "lopozz/JaQuADRetrieval"
JA_DIR = Path("../data/jaquad_mteb").resolve()
IT_DIR = Path("../data/jaquad_mteb_itquad").resolve()
SPLITS = ("train", "test")

for split in SPLITS:
    for lang, data_dir in (("ja", JA_DIR), ("it", IT_DIR)):
        for subset in ("corpus", "queries", "qrels"):
            ds = Dataset.from_parquet(str(data_dir / f"{split}_{subset}.parquet"))
            ds.push_to_hub(TARGET_REPO_ID, config_name=f"{lang}-{subset}", split=split)

print(f"Pushed to: https://huggingface.co/datasets/{TARGET_REPO_ID}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


... (57 outputs left out)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to: https://huggingface.co/datasets/lopozz/JaQuADRetrieval


## Dataset Merger

In [ ]:
from pathlib import Path
from lm_toolkit.splade import DatasetSpec, combine_datasets
split = "train"
combined = combine_datasets(
    [
        DatasetSpec("mmarco_it_75k", max_tokens=510),
        DatasetSpec("germandpr_itquad", max_tokens=510),
    ],
    split=split,
)


combined_output_dir = Path("../data/mmarco_germandpr_itquad_combined")
combined_output_dir.mkdir(parents=True, exist_ok=True)

combined["corpus"].to_parquet(str(combined_output_dir / f"{split}_corpus.parquet"))
combined["queries"].to_parquet(str(combined_output_dir / f"{split}_queries.parquet"))
combined["qrels"].to_parquet(str(combined_output_dir / f"{split}_qrels.parquet"))

[mmarco_it_75k] train: 106165 documents, 75000 queries, 150000 qrels


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (548 > 512). Running this sequence through the model will result in indexing errors


Filter:   0%|          | 0/24009 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9275 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9275 [00:00<?, ? examples/s]

[germandpr_itquad] train: 23528 documents, 8653 queries, 8653 qrels


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

12063862

# Train

In [ ]:
from datasets import load_dataset

repo_id = "lopozz/CulturaViva-Retrieval"


def build_pair_dataset(
    repo_id: str, split: str, max_examples: int | None = None
) -> Dataset:
    queries_ds = load_dataset(repo_id, "queries", split=split)
    corpus_ds = load_dataset(repo_id, "corpus", split=split)
    qrels_ds = load_dataset(repo_id, "qrels", split=split)

    query_text_by_id = {row["_id"]: row["text"] for row in queries_ds}
    doc_text_by_id = {row["_
id"]: row["text"] for row in corpus_ds}

    pairs = []
    for row in qrels_ds:
        if int(row["score"]) > 0:
            qid = row["query-id"]
            did = row["corpus-id"]

            if qid in query_text_by_id and did in doc_text_by_id:
                pairs.append(
                    {
                        "query": query_text_by_id[qid],
                        "answer": doc_text_by_id[did],
                    }
                )

                if max_examples is not None and len(pairs) >= max_examples:
                    break

    return Dataset.from_list(pairs)


train_dataset = build_pair_dataset(repo_id, "train", max_examples=100)
eval_dataset = build_pair_dataset(repo_id, "test", max_examples=10)

print(train_dataset)
print(eval_dataset)

query_ids = set(queries_dataset["_id"])
corpus_ids = set(corpus_dataset["_id"])

missing_query_ids = {
    query_id
    for query_id in qrels_dataset["query-id"]
    if query_id not in query_ids
}

missing_document_ids = {
    document_id
    for document_id in qrels_dataset["corpus-id"]
    if document_id not in corpus_ids
}

print("Queries:", len(queries_dataset))
print("Corpus documents:", len(corpus_dataset))
print("Positive qrels:", len(qrels_dataset))
print("Missing qrel queries:", len(missing_query_ids))
print("Missing relevant documents:", len(missing_document_ids))

assert not missing_query_ids
assert not missing_document_ids
assert relevant_doc_ids.issubset(corpus_ids)

print("Validation passed.")



In [ ]:
import logging

from sentence_transformers import (
    SparseEncoder,
    SparseEncoderModelCardData,
    SparseEncoderTrainer,
    SparseEncoderTrainingArguments,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.losses import (
    SparseMultipleNegativesRankingLoss,
    SpladeLoss,
)
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO
)

# 1. Load a model to finetune with 2. (Optional) model card data
mlm_transformer = Transformer("jhu-clsp/mmBERT-small", transformer_task="fill-mask")
vocab_dim = len(mlm_transformer.tokenizer)

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=vocab_dim,
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)


# 4. Define a loss function
loss = SpladeLoss(
    model=model,
    loss=SparseMultipleNegativesRankingLoss(model=model),
    query_regularizer_weight=0,
    document_regularizer_weight=3e-3,
)

batch_size = 1
# 5. (Optional) Specify training arguments
run_name = "inference-free-splade-mmBERT-small-ita"
args = SparseEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    learning_rate_mapping={
        r"0\.sub_modules\.query\.0\.weight": 1e-3
    },  # Set a higher learning rate for the SparseStaticEmbedding module (see https://huggingface.co/blog/train-sparse-encoder#inference-free-splade)
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    router_mapping={
        "query": "query",
        "answer": "document",
    },  # Map the column names to the routes
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
)

# 6. Create a trainer & train
trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

# 9. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

# 10. (Optional) Push it to the Hugging Face Hub
# model.push_to_hub(run_name)

In [2]:
from sentence_transformers import SparseEncoder
import torch

model = SparseEncoder("nickprock/splade-bert-base-italian-xxl-uncased-cv")

def show_sparse_terms(text, top_k=30):
    emb = model.encode([text], convert_to_tensor=True)

    # SparseEncoder may return a sparse tensor
    row = emb[0]

    if row.is_sparse:
        row = row.to_dense()

    values, indices = torch.topk(row, k=top_k)

    vocab = model.tokenizer.get_vocab()
    inv_vocab = {v: k for k, v in vocab.items()}

    return [
        (inv_vocab[int(idx)], float(score))
        for idx, score in zip(indices, values)
        if float(score) > 0
    ]

print(show_sparse_terms("effetti collaterali tachipirina nei bambini"))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11614.27it/s]


[('##pi', 0.9616959095001221), ('##rina', 0.4469548761844635), ('tachi', 0.43464091420173645), ('bambini', 0.42675817012786865), ('effetto', 0.011501946486532688)]


# Evaluate

In [1]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sparse_encoder.evaluation import (
    SparseInformationRetrievalEvaluator,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 1) Load Italian subsets
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
params = {"path": "mteb/WikipediaRetrievalMultilingual", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
# params = {"path": "mteb/WebFAQRetrieval", "split": "test"}
# names = ["ita-corpus", "ita-queries", "ita-qrels"]
# params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
names = ["corpus", "queries", "qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)

# 4) Build evaluator
ir_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    batch_size=16,
    accuracy_at_k=[1],
    write_predictions=True
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "../models/splade-bert-base-italian-xxl-uncased-cv"
model_path = "../models/splade-bert-base-italian-uncased-cv"



# CHOOSE ONE OPTION:
model = SparseEncoder(model_path)

# OR

# mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

# splade_pooling = SpladePooling(
#     pooling_strategy="max",
#     embedding_dimension=mlm_transformer.get_embedding_dimension(),
# )
# router = Router.for_query_document(
#     # query_modules=[
#     #     SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
#     # ], # this doesn't work with nickprock
#     query_modules=[mlm_transformer, splade_pooling],
#     document_modules=[mlm_transformer, splade_pooling],
# )

# model = SparseEncoder(
#     modules=[router],
#     model_card_data=SparseEncoderModelCardData(
#         language="it",
#         license="apache-2.0",
#         model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
#     ),
# )

model_name = f'{model_path.split('/')[0]}__{model_path.split('/')[1]}'


/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5710.11it/s]


In [2]:
# 6) Run evaluation
results = ir_evaluator(model)
print(results)

Corpus Chunks: 100%|██████████| 1/1 [02:22<00:00, 142.74s/it]


{'dot_accuracy@1': 0.9551070123542718, 'dot_precision@1': 0.9551070123542718, 'dot_precision@3': 0.32956325039150863, 'dot_precision@5': 0.198886375500261, 'dot_precision@10': 0.09979119540629894, 'dot_recall@1': 0.9551070123542718, 'dot_recall@3': 0.9886897511745258, 'dot_recall@5': 0.994431877501305, 'dot_recall@10': 0.9979119540629894, 'dot_ndcg@10': 0.9787915571907403, 'dot_mrr@10': 0.9724042357503289, 'dot_map@100': 0.9725030408786961, 'query_active_dims': 50.3782844543457, 'query_sparsity_ratio': 0.998380223636604, 'corpus_active_dims': 92.2392578125, 'corpus_sparsity_ratio': 0.99703429818621, 'avg_flops': 1.5672399997711182}


## Token Retention Rate (TRR)

The **Token Retention Rate (TRR)** is a diagnostic metric for sparse neural retrieval models. It quantifies the proportion of semantically significant tokens from an original input sequence that survive the model's sparsity regularization and receive a positive weight in the final latent representation.

#### Mathematical Formulation

Let $x$ be an input text sequence (a query or a document).

Let $\mathcal{V}$ represent the complete vocabulary of the model's tokenizer.

Let $T(x) \subset \mathcal{V}$ denote the set of unique subword tokens extracted from $x$ by the tokenizer.

Let $\mathcal{U} \subset \mathcal{V}$ denote the set of uninformative tokens, specifically defined as the union of standard stopwords, punctuation marks, and model-specific special tokens (e.g., `[CLS]`, `[SEP]`).

We define the set of **relevant original tokens**, $R(x)$, as the relative complement of $\mathcal{U}$ in $T(x)$:

$$ R(x) = T(x) \setminus \mathcal{U} $$

Let $E(x) \in \mathbb{R}^{|\mathcal{V}|}$ be the sparse vector representation of $x$ produced by the neural encoder. We define $S(x)$ as the support set of this representation, representing all vocabulary terms that the model activated with a weight strictly greater than zero:

$$ S(x) = { v \in \mathcal{V} \mid E(x)_v > 0 } $$

The **Token Retention Rate** for a single sequence $x$ is defined as the intersection of the relevant original tokens and the activated sparse tokens, divided by the total number of relevant original tokens:

$$ \text{TRR}(x) = \frac{| R(x) \cap S(x) |}{| R(x) |} $$

*Note: In the boundary condition where an input text consists entirely of stopwords or punctuation ($|R(x)| = 0$), $\text{TRR}(x)$ is mathematically undefined and conventionally omitted from aggregation.*

#### Corpus-Level Aggregation

When evaluating a model across an entire dataset $X$ (such as a corpus of documents or a set of queries), the metric is typically reported as the macro-averaged expected value:

$$ \text{TRR}(X) = \frac{1}{|X|} \sum_{x \in X} \frac{| R(x) \cap S(x) |}{| R(x) |} $$

#### Interpretation

* **$\text{TRR}(x) \to 1.0$:** Indicates high lexical fidelity. The model preserves the exact terminology of the input sequence, functioning similarly to traditional BM25 but with reweighted terms and added expansions.
* **$\text{TRR}(x) \to 0.0$:** Indicates extreme vocabulary pruning or aggressive semantic translation. The model has discarded the original vocabulary in favor of mapping the concept entirely to distinct expansion terms within the latent space. In practice, an unexpectedly low TRR often signals that the model's sparsity penalty (e.g., the FLOPS regularizer in SPLADE) is too aggressive, resulting in the loss of critical exact-match anchors.

In [3]:
import string

stop_words_it = {
    "a", "abbia", "abbiamo", "abbiano", "abbiate", "ad", "agl", "agli", "ai", 
    "al", "all", "alla", "alle", "allo", "anche", "avemmo", "avendo", "avesse", 
    "avessero", "avessi", "avessimo", "aveste", "avesti", "avete", "aveva", 
    "avevamo", "avevano", "avevate", "avevi", "avevo", "avrai", "avranno", 
    "avrebbe", "avrebbero", "avrei", "avremmo", "avremo", "avreste", "avresti", 
    "avrete", "avrò", "avuta", "avute", "avuti", "avuto", "c", "che", "chi", 
    "ci", "coi", "col", "come", "con", "contro", "cui", "da", "dagl", "dagli", 
    "dai", "dal", "dall", "dalla", "dalle", "dallo", "degl", "degli", "dei", 
    "del", "dell", "della", "delle", "dello", "di", "dov", "dove", "e", "ebbe", 
    "ebbero", "ebbi", "ed", "era", "erano", "eravamo", "eravate", "eri", "ero", 
    "essendo", "faccia", "facciamo", "facciano", "facciate", "faccio", "facemmo", 
    "facendo", "facesse", "facessero", "facessi", "facessimo", "faceste", 
    "facesti", "faceva", "facevamo", "facevano", "facevate", "facevi", "facevo", 
    "fai", "fanno", "farai", "faranno", "farebbe", "farebbero", "farei", 
    "faremmo", "faremo", "fareste", "faresti", "farete", "farò", "fece", 
    "fecero", "feci", "fosse", "fossero", "fossi", "fossimo", "foste", "fosti", 
    "fu", "fui", "fummo", "furono", "gli", "ha", "hai", "hanno", "ho", "i", "il", 
    "in", "io", "l", "la", "le", "lei", "li", "lo", "loro", "lui", "ma", "mi", 
    "mia", "mie", "miei", "mio", "ne", "negl", "negli", "nei", "nel", "nell", 
    "nella", "nelle", "nello", "no", "noi", "non", "nostra", "nostre", "nostri", 
    "nostro", "o", "per", "perché", "più", "quale", "quanta", "quante", "quanti", 
    "quanto", "quella", "quelle", "quelli", "quello", "questa", "queste", 
    "questi", "questo", "sarai", "saranno", "sarebbe", "sarebbero", "sarei", 
    "saremmo", "saremo", "sareste", "saresti", "sarete", "sarò", "se", "sei", 
    "si", "sia", "siamo", "siano", "siate", "siete", "sono", "sta", "stai", 
    "stando", "stanno", "starai", "staranno", "starebbe", "starebbero", "starei", 
    "staremmo", "staremo", "stareste", "staresti", "starete", "starò", "stata", 
    "state", "stati", "stato", "stava", "stavamo", "stavano", "stavate", "stavi", 
    "stavo", "stemmo", "stesse", "stessero", "stessi", "stessimo", "steste", 
    "stesti", "stette", "stettero", "stetti", "sto", "su", "sua", "sue", "sugl", 
    "sugli", "sui", "sul", "sull", "sulla", "sulle", "sullo", "suo", "suoi", "ti", 
    "tra", "tu", "tua", "tue", "tuo", "tuoi", "tutta", "tutte", "tutti", "tutto", 
    "un", "una", "uno", "vi", "voi", "vostra", "vostre", "vostri", "vostro"
}
punctuation = set(string.punctuation)


In [ ]:
def is_relevant_token(token: str) -> bool:
    """Checks if a token is a stopword or punctuation."""
    # Clean the WordPiece '##' prefix for accurate stopword matching
    clean_token = token.lower()
    
    if not clean_token:
        return False
    if clean_token in stop_words_it:
        return False
    if all(char in punctuation for char in clean_token):
        return False
    return True

def extract_retained_tokens(sparse_rep, tokenizer):
    """
    Converts a sparse representation returned by SparseEncoder.encode
    into a set of retained tokenizer tokens.
    Handles dict, sparse tensor, and dense tensor outputs.
    """

    if hasattr(sparse_rep, "detach"):
        sparse_rep = sparse_rep.detach().cpu()

        # Sparse tensor
        if sparse_rep.is_sparse:
            sparse_rep = sparse_rep.coalesce()
            token_ids = sparse_rep.indices()

            # If shape is [vocab_size], indices shape is [1, nnz]
            # If shape is [batch, vocab_size], indices shape is [2, nnz]
            if token_ids.shape[0] == 1:
                token_ids = token_ids[0]
            else:
                token_ids = token_ids[-1]

            token_ids = token_ids.tolist()
            return set(tokenizer.convert_ids_to_tokens(token_ids))

        # Dense tensor
        token_ids = sparse_rep.nonzero(as_tuple=True)[0].tolist()
        return set(tokenizer.convert_ids_to_tokens(token_ids))

    raise TypeError(f"Unsupported sparse representation type: {type(sparse_rep)}")

def calculate_retention_ratio(texts, sparse_representations, tokenizer):
    """Calculates the average ratio of retained relevant tokens."""

    retention_ratios = []

    for text, sparse_rep in zip(texts, sparse_representations):
        orig_tokens = tokenizer.tokenize(text)

        orig_relevant = set(
            tok for tok in orig_tokens
            if is_relevant_token(tok) and tok not in tokenizer.all_special_tokens
        )

        if not orig_relevant:
            continue

        retained_tokens = extract_retained_tokens(sparse_rep, tokenizer)

        if len(retained_tokens) == 0:
            retention_ratios.append(0.0)
            continue

        overlap = orig_relevant.intersection(retained_tokens)
        ratio = len(overlap) / len(orig_relevant)
        retention_ratios.append(ratio)

    return sum(retention_ratios) / len(retention_ratios) if retention_ratios else 0.0

In [13]:
# --- RUNNING THE CUSTOM EVALUATION ---
print("\n--- Token Retention Evaluation ---")

# Access the tokenizer from your Transformer module
tokenizer = mlm_transformer.tokenizer
batch_size = 50
# 1. Evaluate Query Retention
query_texts = list(queries.values())
print("Calculating Retention for Queries...")
sparse_representations = model.encode(query_texts, batch_size=batch_size, show_progress_bar=True)
avg_query_retention = calculate_retention_ratio(query_texts, sparse_representations, tokenizer)

# 2. Evaluate Relevant Document Retention
# Fetch the texts of all documents that are considered relevant to at least one query
relevant_doc_ids = set()
for doc_ids in relevant_docs.values():
    relevant_doc_ids.update(doc_ids)

# Filter out doc_ids that might be in qrels but missing from the corpus (sanity check)
relevant_doc_texts = [corpus[doc_id] for doc_id in relevant_doc_ids if doc_id in corpus]
sparse_representations = model.encode(relevant_doc_texts, batch_size=batch_size, show_progress_bar=True)

print("\nCalculating Retention for Relevant Documents...")
# avg_doc_retention = calculate_retention_ratio(relevant_doc_texts, sparse_representations, tokenizer)
print(f"\nFinal Metric - Average Query Token Retention Ratio: {avg_query_retention:.4f}")
# print(f"Final Metric - Average Document Token Retention Ratio: {avg_doc_retention:.4f}")


--- Token Retention Evaluation ---
Calculating Retention for Queries...


Batches: 100%|██████████| 30/30 [00:09<00:00,  3.11it/s]



Calculating Retention for Relevant Documents...

Final Metric - Average Query Token Retention Ratio: 0.8408
Final Metric - Average Document Token Retention Ratio: 0.6651


In [ ]:
# lopozz/CulturaViva-Retrieval
# Final Metric - Average Query Token Retention Ratio: 0.7165
# Final Metric - Average Document Token Retention Ratio: 0.2025

# mteb/MuPLeR-retrieval
# Final Metric - Average Query Token Retention Ratio: 0.8544
# Final Metric - Average Document Token Retention Ratio: 0.5398

# mteb/WebFAQRetrieval
# Final Metric - Average Query Token Retention Ratio: 0.7760
# Final Metric - Average Document Token Retention Ratio: 0.6884

# mteb/WikipediaRetrievalMultilingual
# Final Metric - Average Query Token Retention Ratio: 0.8408
# Final Metric - Average Document Token Retention Ratio: 0.6651



## MTEB Benchmark

### Load Tasks

In [ ]:
import yaml
import mteb

with open("../configs/ds/mteb-retrieval-ita.yml", "r") as f:
    config = yaml.safe_load(f)

excluded = set(config["excluded_tasks"])

tasks = mteb.get_tasks(
    languages=config["languages"],
    modalities=config["modalities"],
    task_types=config["task_types"],
    exclusive_language_filter=True,
    eval_splits=["test"],
)

tasks = [t for t in tasks if t.metadata.name not in excluded]
tasks[:1]

[MuPLeRRetrieval(name='MuPLeR-retrieval', languages=['ita'])]

In [13]:
from mteb.abstasks.retrieval import AbsTaskRetrieval
from mteb.abstasks.task_metadata import TaskMetadata


class CulturaVivaRetrieval(AbsTaskRetrieval):
    """Custom task: not registered in MTEB, hosted outside the mteb/ org."""

    metadata = TaskMetadata(
        name="CulturaViva-Retrieval",
        description="Italian retrieval dataset covering culturally-grounded, long-form generated content.",
        reference="https://huggingface.co/datasets/lopozz/CulturaViva-Retrieval",
        dataset={
            "path": "lopozz/CulturaViva-Retrieval",
            "revision": "2347b286719a879cc5129cea5c4c5d3fa813dd1b",
        },
        type="Retrieval",
        category="t2t",
        modalities=["text"],
        eval_splits=["test"],
        eval_langs=["ita-Latn"],
        main_score="ndcg_at_10",
    )


tasks.append(CulturaVivaRetrieval())
tasks[-1]

CulturaVivaRetrieval(name='CulturaViva-Retrieval', languages=['ita'])

### Performance Table

In [14]:
import mteb

model_names = [
    "mteb/baseline-bm25s",
    "nickprock/splade-bert-base-italian-xxl-uncased-cv",
    "google/embeddinggemma-300m",
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1",
    "models/splade-bert-base-italian-uncased-cv",
    "models/splade-bert-base-italian-xxl-uncased-cv"
]
cache = mteb.ResultCache("..")
results = cache.load_results(models=model_names, tasks=tasks, require_model_meta=False)

results.to_dataframe()

model_name,task_name,google/embeddinggemma-300m,models/splade-bert-base-italian-uncased-cv,models/splade-bert-base-italian-xxl-uncased-cv,mteb/baseline-bm25s,nickprock/splade-bert-base-italian-xxl-uncased-cv,opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
0,CulturaViva-Retrieval,0.98935,NaN,0.981895,0.988703,0.723767,NaN
1,MuPLeR-retrieval,0.84454,0.099159,0.549873,0.785970,0.510880,0.551909
2,WebFAQRetrieval,0.79523,NaN,0.265128,0.559180,0.517111,0.554244
3,WikipediaRetrievalMultilingual,0.91761,NaN,0.658576,0.843600,0.799550,0.837200


### Single Analysis